# 01. Extracción de datos de Chartmetric

Construye las trayectorias mensuales de los artistas de la muestra (enero 2021 a marzo 2026) con dos endpoints de la API de Chartmetric: **Fan Stats** (audiencias por plataforma) y **Playlist Evolution** (presencia en playlists).

- Las funciones de extracción son *cache-first*: antes de cada request revisan el cache local y el log, y guardan la respuesta cruda antes de procesarla.
- `EJECUTAR_EXTRACCION_API` (sección 1) controla todas las celdas que hacen requests. Con `FALSE`, el notebook corre completo desde el cache, sin consumir cuota.
- Licencia: los datos y los registros extraídos no se publican. El código se conserva por replicabilidad.

**Salidas:** dataset mensual consolidado (`chartmetric_pilotos/`), `artistas_ST_raw_<n>_long.csv` y `_wide.csv`, e insumos del notebook 02.

In [ ]:
# PAQUETES

paquetes <- c("httr", "jsonlite", "data.table", "lubridate")

faltan <- paquetes[!paquetes %in% rownames(installed.packages())]

if (length(faltan) > 0) {
  install.packages(faltan)
}

library(httr)
library(jsonlite)
library(data.table)
library(lubridate)

## 1. Configuración

In [ ]:
# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

BASE_URL <- "https://api.chartmetric.com"

# Interruptor de las celdas que hacen requests a la API
# (token, Fan Stats por tramos, recuperación de 99 artistas y
# Playlist Evolution). FALSE: todo se lee del cache local.
EJECUTAR_EXTRACCION_API <- FALSE

# Período definitivo para las nuevas extracciones.
FAN_SINCE <- as.Date("2021-01-01")
END_DATE <- as.Date("2026-03-31")

FAN_SOURCES <- c(
  "spotify",
  "twitter",
  "youtube_channel",
  "youtube_artist",
  "instagram",
  "tiktok"
)

TARGET_FAN_METRICS <- list(
  spotify         = c("followers", "listeners"),
  twitter         = c("followers"),
  youtube_channel = c("subscribers", "views"),
  youtube_artist  = c("monthly_views"),
  instagram       = c("followers"),
  tiktok          = c("followers", "likes")
)

TARGET_PLAYLIST_METRICS <- list(
  spotify = c(
    "playlist_count",
    "ed_playlist_count",
    "playlist_total_reach",
    "ed_playlist_total_reach"
  ),
  youtube = c(
    "playlist_count",
    "ed_playlist_count",
    "playlist_total_reach",
    "ed_playlist_total_reach"
  ),
  amazon = c(
    "playlist_count",
    "ed_playlist_count"
  )
)

OUT_DIR <- "chartmetric_pilotos"

dir.create(
  OUT_DIR,
  showWarnings = FALSE,
  recursive = TRUE
)

# ------------------------------------------------------------
# Archivos LEGACY
# Se conservan intactos.
# ------------------------------------------------------------

DATA_FILE <- file.path(
  OUT_DIR,
  "chartmetric_raw_long.rds"
)

RAW_RESPONSE_FILE <- file.path(
  OUT_DIR,
  "chartmetric_raw_responses.rds"
)

# ------------------------------------------------------------
# Log general
# ------------------------------------------------------------

LOG_FILE <- file.path(
  OUT_DIR,
  "chartmetric_request_log.csv"
)

# ------------------------------------------------------------
# Nueva arquitectura
# Un archivo por request
# ------------------------------------------------------------

RAW_REQUEST_DIR <- file.path(
  OUT_DIR,
  "raw_requests_2021_2026"
)

MONTHLY_REQUEST_DIR <- file.path(
  OUT_DIR,
  "monthly_requests_2021_2026"
)

dir.create(
  RAW_REQUEST_DIR,
  showWarnings = FALSE,
  recursive = TRUE
)

dir.create(
  MONTHLY_REQUEST_DIR,
  showWarnings = FALSE,
  recursive = TRUE
)

# Dataset mensual consolidado.
MONTHLY_FILE <- file.path(
  OUT_DIR,
  "chartmetric_monthly_2021_2026.rds"
)

cat(
  "Período:",
  format(FAN_SINCE),
  "a",
  format(END_DATE),
  "\n"
)

cat(
  "Raw legacy:",
  DATA_FILE,
  "\n"
)

cat(
  "Requests crudos nuevos:",
  RAW_REQUEST_DIR,
  "\n"
)

cat(
  "Requests mensuales nuevos:",
  MONTHLY_REQUEST_DIR,
  "\n"
)

cat(
  "Mensual consolidado:",
  MONTHLY_FILE,
  "\n"
)

## 2. Archivo maestro de artistas

In [ ]:
# ARCHIVO MAESTRO: dataset_raw_concat_7000

artist_candidates <- c(
  "dataset_raw_concat_7000.csv"
)

existing_candidates <- artist_candidates[file.exists(artist_candidates)]

ARTIST_FILE <- if (length(existing_candidates) > 0) {
  existing_candidates[1]
} else {
  NA_character_
}

load_artist_catalog <- function(path = ARTIST_FILE) {

  if (is.na(path) || !file.exists(path)) {
    message(
      "dataset_raw_concat_7000 no está en el working directory. ",
      "Definir ARTIST_FILE manualmente."
    )
    return(data.table())
  }

  x <- fread(
    path,
    select = c(
      "chartmetric_id",
      "artist_name",
      "cm_artist_rank"
    )
  )

  x[, ID := as.character(chartmetric_id)]

  x <- x[
    ,
    .(
      ID,
      artist_name,
      cm_artist_rank
    )
  ]

  unique(x, by = "ID")
}

ARTISTS <- load_artist_catalog()

if (nrow(ARTISTS) > 0) {
  cat("Artistas cargados:", nrow(ARTISTS), "\n")
  print(head(ARTISTS))
}

## 3. Autenticación

In [ ]:
# AUTENTICACIÓN
# Reutiliza el access token hasta que falte aproximadamente 1 minuto para vencer.

.cm_auth <- new.env(parent = emptyenv())
.cm_auth$token <- NULL
.cm_auth$expires_at <- as.POSIXct(NA)

get_access_token <- function(force = FALSE) {

  refresh_token <- Sys.getenv("CHARTMETRIC_REFRESH_TOKEN")

  if (refresh_token == "") {
    stop(
      "Falta CHARTMETRIC_REFRESH_TOKEN en .Renviron. ",
      "Agregalo y reiniciá el kernel de R."
    )
  }

  ahora <- Sys.time()

  vigente <- (
    !force &&
    !is.null(.cm_auth$token) &&
    !is.na(.cm_auth$expires_at) &&
    ahora < (.cm_auth$expires_at - 60)
  )

  if (vigente) {
    return(.cm_auth$token)
  }

  resp <- POST(
    paste0(BASE_URL, "/api/token"),
    body = list(refreshtoken = refresh_token),
    encode = "json",
    timeout(60)
  )

  status <- status_code(resp)
  texto <- content(resp, as = "text", encoding = "UTF-8")

  if (status != 200) {
    stop(
      "No se pudo obtener access token. HTTP ",
      status,
      "\n",
      substr(texto, 1, 1000)
    )
  }

  body <- fromJSON(
    texto,
    simplifyVector = TRUE
  )

  if (is.null(body$token) || is.null(body$expires_in)) {
    stop("La respuesta de autenticación no tiene la estructura esperada.")
  }

  .cm_auth$token <- body$token
  .cm_auth$expires_at <- ahora + as.numeric(body$expires_in)

  cat(
    "Autenticación OK. Access token obtenido; no se muestra.\n"
  )

  invisible(.cm_auth$token)
}

In [ ]:
if (EJECUTAR_EXTRACCION_API) {

  # PRUEBA DE VIGENCIA DE LA CUENTA/API
  # No descarga datos de artistas.

  get_access_token(force = TRUE)

} else {

  cat("Extraccion desactivada (EJECUTAR_EXTRACCION_API = FALSE): sin requests a la API.\n")
}

## 4. Helpers, persistencia, cache y log

In [ ]:
# HELPERS BÁSICOS

scalar_chr <- function(x) {
  if (is.null(x) || length(x) == 0) {
    return(NA_character_)
  }

  as.character(
    unlist(
      x,
      recursive = TRUE,
      use.names = FALSE
    )[1]
  )
}

scalar_num <- function(x) {
  if (is.null(x) || length(x) == 0) {
    return(NA_real_)
  }

  suppressWarnings(
    as.numeric(
      unlist(
        x,
        recursive = TRUE,
        use.names = FALSE
      )[1]
    )
  )
}

scalar_lgl <- function(x) {
  if (is.null(x) || length(x) == 0) {
    return(NA)
  }

  suppressWarnings(
    as.logical(
      unlist(
        x,
        recursive = TRUE,
        use.names = FALSE
      )[1]
    )
  )
}

# Chartmetric puede devolver tanto "YYYY-MM-DD" como timestamps ISO con milisegundos.
parse_timestamp <- function(x) {

  x <- as.character(x)

  suppressWarnings(
    as.POSIXct(
      x,
      tz = "UTC",
      tryFormats = c(
        "%Y-%m-%dT%H:%M:%OSZ",
        "%Y-%m-%dT%H:%M:%OS",
        "%Y-%m-%d %H:%M:%OS",
        "%Y-%m-%d"
      )
    )
  )
}

safe_min_date <- function(x) {
  x <- x[!is.na(x)]
  if (length(x) == 0) return(NA_character_)
  format(min(x))
}

safe_max_date <- function(x) {
  x <- x[!is.na(x)]
  if (length(x) == 0) return(NA_character_)
  format(max(x))
}

In [ ]:
# ============================================================
# CACHE DE RESPUESTAS CRUDAS
# UN ARCHIVO INDEPENDIENTE POR REQUEST
# ============================================================

request_file_name <- function(request_key) {

  safe_name <- gsub(
    "|",
    "__",
    as.character(request_key),
    fixed = TRUE
  )

  paste0(
    safe_name,
    ".rds"
  )
}


raw_response_path <- function(request_key) {

  file.path(
    RAW_REQUEST_DIR,
    request_file_name(request_key)
  )
}


monthly_response_path <- function(request_key) {

  file.path(
    MONTHLY_REQUEST_DIR,
    request_file_name(request_key)
  )
}


raw_response_exists <- function(request_key) {

  file.exists(
    raw_response_path(request_key)
  )
}


monthly_response_exists <- function(request_key) {

  file.exists(
    monthly_response_path(request_key)
  )
}


get_raw_response <- function(request_key) {

  path <- raw_response_path(
    request_key
  )

  if (!file.exists(path)) {
    return(NULL)
  }

  safe_read_rds(
    path,
    default = NULL
  )
}


save_raw_response <- function(
  request_key,
  body
) {

  path <- raw_response_path(
    request_key
  )

  if (file.exists(path)) {
    return(invisible(NULL))
  }

  safe_save_rds(
    body,
    path
  )

  invisible(NULL)
}


save_monthly_response <- function(
  request_key,
  monthly
) {

  safe_save_rds(
    monthly,
    monthly_response_path(request_key)
  )

  invisible(NULL)
}

In [ ]:
# ============================================================
# LECTURA Y GUARDADO SEGURO DE ARCHIVOS RDS
# Versión robusta para Windows
# ============================================================


safe_read_rds <- function(path, default = NULL) {

  backup <- paste0(path, ".bak")

  if (!file.exists(path)) {

    if (file.exists(backup)) {

      ok <- file.copy(
        backup,
        path,
        overwrite = TRUE
      )

      if (!ok) {
        stop(
          "No se pudo recuperar el backup de: ",
          path
        )
      }

    } else {

      return(default)
    }
  }


  tryCatch(

    readRDS(path),

    error = function(e) {

      if (!file.exists(backup)) {
        stop(conditionMessage(e))
      }

      ok <- file.copy(
        backup,
        path,
        overwrite = TRUE
      )

      if (!ok) {
        stop(
          "No se pudo recuperar el backup de: ",
          path
        )
      }

      readRDS(path)
    }
  )
}


safe_save_rds <- function(object, path) {

  dir.create(
    dirname(path),
    recursive = TRUE,
    showWarnings = FALSE
  )

  tmp <- paste0(
    path,
    ".tmp"
  )

  backup <- paste0(
    path,
    ".bak"
  )


  # Limpiar temporal viejo.
  if (file.exists(tmp)) {
    unlink(tmp)
  }


  # ----------------------------------------------------------
  # 1. Escribir primero el temporal
  # ----------------------------------------------------------

  saveRDS(
    object,
    tmp
  )


  # ----------------------------------------------------------
  # 2. Verificar el temporal
  # ----------------------------------------------------------

  valid_tmp <- tryCatch(
    {
      readRDS(tmp)
      TRUE
    },
    error = function(e) FALSE
  )


  if (!valid_tmp) {

    unlink(tmp)

    stop(
      "El archivo temporal RDS quedó inválido."
    )
  }


  # ----------------------------------------------------------
  # 3. Crear backup si ya existe archivo definitivo
  # ----------------------------------------------------------

  if (file.exists(backup)) {
    unlink(backup)
  }


  if (file.exists(path)) {

    ok_backup <- file.copy(
      path,
      backup,
      overwrite = TRUE
    )

    if (!ok_backup) {

      unlink(tmp)

      stop(
        "No se pudo crear el backup. ",
        "El archivo original no fue modificado."
      )
    }
  }


  # ----------------------------------------------------------
  # 4. Copiar temporal al archivo definitivo
  # ----------------------------------------------------------

  ok_save <- file.copy(
    tmp,
    path,
    overwrite = TRUE
  )


  if (!ok_save) {

    unlink(tmp)

    stop(
      "No se pudo guardar el nuevo archivo RDS."
    )
  }


  unlink(tmp)


  # ----------------------------------------------------------
  # 5. Verificación final
  # ----------------------------------------------------------

  valid_final <- tryCatch(
    {
      readRDS(path)
      TRUE
    },
    error = function(e) FALSE
  )


  if (!valid_final) {

    if (file.exists(backup)) {

      file.copy(
        backup,
        path,
        overwrite = TRUE
      )
    }

    stop(
      "El nuevo RDS quedó inválido. ",
      "Se restauró la versión anterior."
    )
  }


  # ----------------------------------------------------------
  # 6. Todo salió bien
  # ----------------------------------------------------------

  if (file.exists(backup)) {
    unlink(backup)
  }


  invisible(NULL)
}

In [ ]:
# ============================================================
# ARCHIVO LONG ACUMULADO
# ============================================================

load_raw_data <- function() {

  x <- safe_read_rds(
    DATA_FILE,
    default = data.table()
  )

  as.data.table(x)
}


append_raw_data <- function(new_data) {

  if (
    is.null(new_data) ||
    nrow(new_data) == 0
  ) {
    return(invisible(NULL))
  }

  old <- load_raw_data()

  all <- rbindlist(
    list(
      old,
      as.data.table(new_data)
    ),
    fill = TRUE,
    use.names = TRUE
  )

  dedup_cols <- intersect(
    c(
      "artist_id",
      "endpoint",
      "source",
      "metric",
      "timestamp"
    ),
    names(all)
  )

  if (length(dedup_cols) > 0) {

    all <- unique(
      all,
      by = dedup_cols,
      fromLast = TRUE
    )
  }

  safe_save_rds(
    all,
    DATA_FILE
  )

  invisible(NULL)
}

In [ ]:
# ============================================================
# LOG
# Escritura incremental
# ============================================================

LOG_COLUMNS <- c(
  "request_key",
  "artist_id",
  "endpoint",
  "source",
  "requested_since",
  "requested_until",
  "status_code",
  "ok",
  "returned_fields",
  "n_rows",
  "min_date",
  "max_date",
  "rate_limit_limit",
  "rate_limit_remaining",
  "rate_limit_reset",
  "completed_at",
  "note"
)


empty_request_log <- function() {

  as.data.table(
    setNames(
      replicate(
        length(LOG_COLUMNS),
        character(),
        simplify = FALSE
      ),
      LOG_COLUMNS
    )
  )
}


read_request_log <- function() {

  if (!file.exists(LOG_FILE)) {
    return(
      empty_request_log()
    )
  }

  x <- fread(
    LOG_FILE,
    colClasses = "character",
    na.strings = NULL
  )

  for (
    col in setdiff(
      LOG_COLUMNS,
      names(x)
    )
  ) {

    x[
      ,
      (col) := NA_character_
    ]
  }

  setcolorder(
    x,
    LOG_COLUMNS
  )

  # Si un request fue registrado más de una vez,
  # conservar el registro más reciente.
  x <- x[
    !duplicated(
      request_key,
      fromLast = TRUE
    )
  ]

  x
}


write_log_row <- function(row) {

  row <- as.data.table(
    row
  )

  for (
    col in setdiff(
      LOG_COLUMNS,
      names(row)
    )
  ) {

    row[
      ,
      (col) := NA_character_
    ]
  }

  row <- row[
    ,
    ..LOG_COLUMNS
  ]

  for (col in LOG_COLUMNS) {

    row[
      ,
      (col) := as.character(
        get(col)
      )
    ]
  }

  log_exists <-
    file.exists(LOG_FILE) &&
    file.info(LOG_FILE)$size > 0

  fwrite(
    row,
    LOG_FILE,
    append = log_exists,
    col.names = !log_exists,
    na = ""
  )

  invisible(NULL)
}

request_completed <- function(request_key) {

  x <- read_request_log()

  if (nrow(x) == 0) {
    return(FALSE)
  }

  key_value <- as.character(
    request_key
  )

  rows <- x[
    request_key == key_value
  ]

  if (nrow(rows) == 0) {
    return(FALSE)
  }

  any(
    toupper(rows$ok) == "TRUE",
    na.rm = TRUE
  )
}

## 5. Request keys e inventario de requests

In [ ]:
# ============================================================
# REQUEST KEYS Y CONTROL DE PERÍODO
# ============================================================


fan_request_key <- function(
  artist_id,
  source
) {

  paste(
    "fan_stats",
    as.character(artist_id),
    source,
    format(
      FAN_SINCE,
      "%Y-%m-%d"
    ),
    format(
      END_DATE,
      "%Y-%m-%d"
    ),
    "field_default",
    sep = "|"
  )
}


playlist_request_key <- function(
  artist_id
) {

  paste(
    "playlist_evolution",
    as.character(artist_id),
    format(
      FAN_SINCE,
      "%Y-%m-%d"
    ),
    format(
      END_DATE,
      "%Y-%m-%d"
    ),
    "historical_v2",
    sep = "|"
  )
}


fan_completed_for_period <- function(
  log,
  artist_id_value,
  source_value
) {

  if (nrow(log) == 0) {
    return(FALSE)
  }

  rows <- log[
    endpoint == "fan_stats" &
      artist_id == as.character(
        artist_id_value
      ) &
      source == source_value &
      toupper(ok) == "TRUE"
  ]

  if (nrow(rows) == 0) {
    return(FALSE)
  }

  since_dates <- suppressWarnings(
    as.Date(
      rows$requested_since
    )
  )

  until_dates <- suppressWarnings(
    as.Date(
      rows$requested_until
    )
  )

  any(
    !is.na(since_dates) &
      !is.na(until_dates) &
      since_dates <= FAN_SINCE &
      until_dates >= END_DATE
  )
}


fan_needs_api <- function(
  log,
  artist_id,
  source
) {

  if (
    fan_completed_for_period(
      log,
      artist_id,
      source
    )
  ) {

    return(FALSE)
  }

  key <- fan_request_key(
    artist_id,
    source
  )

  if (
    raw_response_exists(
      key
    )
  ) {

    return(FALSE)
  }

  TRUE
}


request_needs_api <- function(
  request_key
) {

  if (
    raw_response_exists(
      request_key
    )
  ) {

    return(FALSE)
  }

  if (
    request_completed(
      request_key
    )
  ) {

    return(FALSE)
  }

  TRUE
}

## 6. GET genérico

In [ ]:
# ============================================================
# GET GENÉRICO
#
# Mantiene exactamente el contrato:
#   ok, status, body, text, headers
#
# 401:
#   renueva el access token una vez.
#
# 429:
#   - espera sólo si el reset es corto;
#   - si la espera supera 10 segundos, NO bloquea el kernel;
#   - la extracción se pausa y podrá retomarse después.
# ============================================================


MAX_SHORT_RATE_LIMIT_WAIT <- 20


header_value <- function(h, name) {

  if (
    is.null(h) ||
    length(h) == 0
  ) {
    return(NA_character_)
  }

  nms <- tolower(
    names(h)
  )

  pos <- which(
    nms == tolower(name)
  )

  if (length(pos) == 0) {
    return(NA_character_)
  }

  as.character(
    h[[pos[1]]]
  )
}


rate_limit_wait <- function(
  h,
  fallback_seconds = 1,
  safety_seconds = 0.25
) {

  reset_value <- suppressWarnings(
    as.numeric(
      header_value(
        h,
        "x-ratelimit-reset"
      )
    )
  )

  if (
    !is.na(reset_value) &&
    is.finite(reset_value)
  ) {

    wait_seconds <-
      reset_value -
      as.numeric(Sys.time()) +
      safety_seconds

    return(
      max(
        wait_seconds,
        safety_seconds
      )
    )
  }

  fallback_seconds
}


rate_limit_reset_at <- function(h) {

  reset_value <- suppressWarnings(
    as.numeric(
      header_value(
        h,
        "x-ratelimit-reset"
      )
    )
  )

  if (
    is.na(reset_value) ||
    !is.finite(reset_value)
  ) {

    return(
      NA_character_
    )
  }

  format(
    as.POSIXct(
      reset_value,
      origin = "1970-01-01",
      tz = "UTC"
    ),
    "%Y-%m-%d %H:%M:%S UTC"
  )
}


cm_get <- function(
  path,
  query = list(),
  max_429_retries = 3L,
  max_short_wait_seconds =
    MAX_SHORT_RATE_LIMIT_WAIT
) {

  token <- get_access_token()


  do_get <- function(tok) {

    tryCatch(
      GET(
        paste0(
          BASE_URL,
          path
        ),
        add_headers(
          Authorization = paste(
            "Bearer",
            tok
          )
        ),
        query = query,
        timeout(90)
      ),
      error = function(e) e
    )
  }


  refreshed_401 <- FALSE
  retry_429 <- 0L


  repeat {

    resp <- do_get(
      token
    )


    # --------------------------------------------------------
    # Error local o de conexión
    # --------------------------------------------------------

    if (
      inherits(
        resp,
        "error"
      )
    ) {

      return(
        list(
          ok = FALSE,
          status = NA_integer_,
          body = NULL,
          text = conditionMessage(resp),
          headers = list()
        )
      )
    }


    status <- status_code(
      resp
    )

    h <- headers(
      resp
    )


    # --------------------------------------------------------
    # 401: renovar token una única vez
    # --------------------------------------------------------

    if (
      status == 401 &&
      !refreshed_401
    ) {

      cat(
        "  401 - renovando access token...\n"
      )

      token <- get_access_token(
        force = TRUE
      )

      refreshed_401 <- TRUE

      next
    }


    # --------------------------------------------------------
    # 429
    # --------------------------------------------------------

    if (status == 429) {

      retry_429 <-
        retry_429 + 1L

      fallback_seconds <- min(
        2 ^ (retry_429 - 1),
        30
      )

      wait_seconds <- rate_limit_wait(
        h,
        fallback_seconds =
          fallback_seconds,
        safety_seconds = 0.25
      )

      reset_at <- rate_limit_reset_at(
        h
      )


      # Espera demasiado larga:
      # NO dejar el kernel bloqueado.
      if (
        wait_seconds >
          max_short_wait_seconds
      ) {

        stop(
          paste0(
            "RATE_LIMIT_LARGO: Chartmetric devolvió 429. ",
            "Indica esperar aproximadamente ",
            round(wait_seconds),
            " segundos. Reset informado: ",
            reset_at,
            "."
          ),
          call. = FALSE
        )
      }


      # Demasiados 429 consecutivos,
      # aunque sean de ventana corta.
      if (
        retry_429 >
          max_429_retries
      ) {

        stop(
          paste0(
            "RATE_LIMIT_REPETIDO: ",
            "se alcanzó el máximo de reintentos 429."
          ),
          call. = FALSE
        )
      }


      cat(
        "  429 corto - esperando ",
        round(
          wait_seconds,
          2
        ),
        " segundos. Reintento ",
        retry_429,
        "/",
        max_429_retries,
        "\n",
        sep = ""
      )

      flush.console()

      Sys.sleep(
        wait_seconds
      )

      next
    }


    # --------------------------------------------------------
    # Respuesta final
    # --------------------------------------------------------

    texto <- content(
      resp,
      as = "text",
      encoding = "UTF-8"
    )


    body <- tryCatch(
      fromJSON(
        texto,
        simplifyVector = FALSE
      ),
      error = function(e) NULL
    )


    return(
      list(
        ok =
          status >= 200 &&
          status < 300,

        status =
          status,

        body =
          body,

        text =
          texto,

        headers =
          h
      )
    )
  }
}

## 7. Fan Stats

In [ ]:
parse_fan_stats <- function(
  body,
  artist_id,
  source
) {

  if (
    is.null(body) ||
    is.null(body$obj)
  ) {

    return(
      data.table()
    )
  }


  obj <- body$obj


  metrics <- TARGET_FAN_METRICS[[source]]


  if (
    is.null(metrics) ||
    length(metrics) == 0
  ) {

    return(
      data.table()
    )
  }


  metrics <- intersect(
    metrics,
    names(obj)
  )


  if (length(metrics) == 0) {

    return(
      data.table()
    )
  }


  resultado <- list()


  for (metric in metrics) {

    pts <- obj[[metric]]


    if (
      is.null(pts) ||
      length(pts) == 0
    ) {

      next
    }


    # Si viene un único punto como objeto,
    # transformarlo en lista de un punto.
    if (
      is.list(pts) &&
      (
        "timestp" %in% names(pts) ||
        "value" %in% names(pts)
      )
    ) {

      pts <- list(
        pts
      )
    }


    if (!is.list(pts)) {
      next
    }


    filas <- lapply(
      pts,
      function(p) {

        if (!is.list(p)) {
          return(NULL)
        }


        if (
          !(
            "timestp" %in%
              names(p)
          )
        ) {

          return(NULL)
        }


        data.table(
          artist_id =
            as.character(
              artist_id
            ),

          endpoint =
            "fan_stats",

          source =
            source,

          metric =
            metric,

          timestamp =
            scalar_chr(
              p$timestp
            ),

          value =
            scalar_num(
              p$value
            ),

          interpolated =
            scalar_lgl(
              p$is_interpolated
            ),

          extrapolated =
            scalar_lgl(
              p$is_extrapolated
            ),

          flags =
            scalar_chr(
              p$flags
            )
        )
      }
    )


    filas <- Filter(
      Negate(is.null),
      filas
    )


    if (
      length(filas) > 0
    ) {

      resultado[[metric]] <- rbindlist(
        filas,
        fill = TRUE
      )
    }
  }


  if (
    length(resultado) == 0
  ) {

    return(
      data.table()
    )
  }


  rbindlist(
    resultado,
    fill = TRUE
  )
}

In [ ]:
# ============================================================
# FAN STATS
# PROCESAMIENTO MENSUAL + EXTRACCIÓN
# ============================================================

fan_raw_to_monthly <- function(raw) {

  if (
    is.null(raw) ||
    nrow(raw) == 0
  ) {

    return(
      data.table()
    )
  }

  d <- copy(
    as.data.table(raw)
  )

  d[
    ,
    timestamp_dt :=
      parse_timestamp(
        timestamp
      )
  ]

  d[
    ,
    observed_date :=
      as.Date(
        timestamp_dt
      )
  ]

  d <- d[
    !is.na(observed_date) &
      observed_date >= FAN_SINCE &
      observed_date <= END_DATE
  ]

  if (nrow(d) == 0) {

    return(
      data.table()
    )
  }

  d[
    ,
    variable := paste(
      endpoint,
      source,
      metric,
      sep = "__"
    )
  ]

  d[
    ,
    month := as.Date(
      format(
        observed_date,
        "%Y-%m-01"
      )
    )
  ]

  setorder(
    d,
    artist_id,
    endpoint,
    source,
    metric,
    month,
    timestamp_dt
  )

  monthly <- d[
    ,
    .SD[.N],
    by = .(
      artist_id,
      endpoint,
      source,
      metric,
      variable,
      month
    )
  ]

  monthly[
    ,
    .(
      artist_id,
      month,
      endpoint,
      source,
      metric,
      variable,
      observed_date,
      value,
      interpolated,
      extrapolated,
      flags
    )
  ]
}


process_fan_body <- function(
  body,
  artist_id,
  source,
  status_code_value = 200L,
  response_headers = list(),
  note = ""
) {

  request_key <- fan_request_key(
    artist_id,
    source
  )

  fields <- if (
    !is.null(body) &&
      !is.null(body$obj)
  ) {

    names(
      body$obj
    )

  } else {

    character()
  }

  raw <- parse_fan_stats(
    body,
    artist_id,
    source
  )

  monthly <- fan_raw_to_monthly(
    raw
  )

  # Guardar el derivado mensual de ESTE request.
  # No se toca el raw_long acumulado.
  save_monthly_response(
    request_key,
    monthly
  )

  fechas <- if (
    nrow(raw) > 0
  ) {

    as.Date(
      parse_timestamp(
        raw$timestamp
      )
    )

  } else {

    as.Date(
      character()
    )
  }

  write_log_row(
    data.table(
      request_key =
        request_key,

      artist_id =
        as.character(
          artist_id
        ),

      endpoint =
        "fan_stats",

      source =
        source,

      requested_since =
        format(
          FAN_SINCE
        ),

      requested_until =
        format(
          END_DATE
        ),

      status_code =
        as.character(
          status_code_value
        ),

      ok =
        "TRUE",

      returned_fields =
        paste(
          fields,
          collapse = ";"
        ),

      n_rows =
        as.character(
          nrow(raw)
        ),

      min_date =
        safe_min_date(
          fechas
        ),

      max_date =
        safe_max_date(
          fechas
        ),

      rate_limit_limit =
        header_value(
          response_headers,
          "x-ratelimit-limit"
        ),

      rate_limit_remaining =
        header_value(
          response_headers,
          "x-ratelimit-remaining"
        ),

      rate_limit_reset =
        header_value(
          response_headers,
          "x-ratelimit-reset"
        ),

      completed_at =
        format(
          Sys.time(),
          "%Y-%m-%d %H:%M:%S"
        ),

      note =
        note
    )
  )

  cat(
    "  Filas objetivo parseadas:",
    nrow(raw),
    "\n"
  )

  cat(
    "  Filas mensuales guardadas:",
    nrow(monthly),
    "\n"
  )

  invisible(
    monthly
  )
}


extract_fan_source <- function(
  artist_id,
  source,
  log_snapshot = NULL
) {

  artist_id <- as.character(
    artist_id
  )

  if (
    !(source %in% FAN_SOURCES)
  ) {

    stop(
      "Source no previsto: ",
      source
    )
  }

  if (is.null(log_snapshot)) {

    log_snapshot <-
      read_request_log()
  }

  request_key <- fan_request_key(
    artist_id,
    source
  )

  # ----------------------------------------------------------
  # 1. Ya existe un request que cubre todo el período.
  # ----------------------------------------------------------

  if (
    fan_completed_for_period(
      log_snapshot,
      artist_id,
      source
    )
  ) {

    # Caso excepcional:
    # existe el raw nuevo pero falta su derivado mensual.
    if (
      raw_response_exists(
        request_key
      ) &&
      !monthly_response_exists(
        request_key
      )
    ) {

      cat(
        "CACHE:",
        source,
        "- reconstruyendo mensual\n"
      )

      body <- get_raw_response(
        request_key
      )

      return(
        process_fan_body(
          body = body,
          artist_id = artist_id,
          source = source,
          status_code_value = 200L,
          response_headers = list(),
          note = "Reconstruido desde cache crudo"
        )
      )
    }

    cat(
      "SKIP:",
      source,
      "- período ya cubierto\n"
    )

    return(
      invisible(NULL)
    )
  }

  # ----------------------------------------------------------
  # 2. Existe respuesta cruda nueva pero faltó procesarla.
  # ----------------------------------------------------------

  if (
    raw_response_exists(
      request_key
    )
  ) {

    cat(
      "CACHE:",
      source,
      "- artista",
      artist_id,
      "\n"
    )

    body <- get_raw_response(
      request_key
    )

    return(
      process_fan_body(
        body = body,
        artist_id = artist_id,
        source = source,
        status_code_value = 200L,
        response_headers = list(),
        note = "Procesado desde cache crudo local"
      )
    )
  }

  # ----------------------------------------------------------
  # 3. Recién acá llamar a Chartmetric.
  # ----------------------------------------------------------

  cat(
    "REQUEST:",
    source,
    "- artista",
    artist_id,
    "\n"
  )

  resp <- cm_get(
    path = paste0(
      "/api/artist/",
      artist_id,
      "/stat/",
      source
    ),
    query = list(
      since =
        format(
          FAN_SINCE
        ),

      until =
        format(
          END_DATE
        )
    )
  )

  if (!resp$ok) {

    write_log_row(
      data.table(
        request_key =
          request_key,

        artist_id =
          artist_id,

        endpoint =
          "fan_stats",

        source =
          source,

        requested_since =
          format(
            FAN_SINCE
          ),

        requested_until =
          format(
            END_DATE
          ),

        status_code =
          as.character(
            resp$status
          ),

        ok =
          "FALSE",

        returned_fields =
          NA_character_,

        n_rows =
          "0",

        min_date =
          NA_character_,

        max_date =
          NA_character_,

        rate_limit_limit =
          header_value(
            resp$headers,
            "x-ratelimit-limit"
          ),

        rate_limit_remaining =
          header_value(
            resp$headers,
            "x-ratelimit-remaining"
          ),

        rate_limit_reset =
          header_value(
            resp$headers,
            "x-ratelimit-reset"
          ),

        completed_at =
          format(
            Sys.time(),
            "%Y-%m-%d %H:%M:%S"
          ),

        note =
          substr(
            resp$text,
            1,
            1000
          )
      )
    )

    cat(
      "  ERROR HTTP:",
      ifelse(
        is.na(resp$status),
        "sin status",
        resp$status
      ),
      "\n"
    )

    return(
      invisible(NULL)
    )
  }

  # Primero preservar la respuesta original.
  save_raw_response(
    request_key,
    resp$body
  )

  # Después procesarla.
  process_fan_body(
    body = resp$body,
    artist_id = artist_id,
    source = source,
    status_code_value = resp$status,
    response_headers = resp$headers,
    note = "Descargado desde API"
  )
}

## 8. Playlist Evolution

In [ ]:
parse_playlist_evolution <- function(body, artist_id) {

  if (
    is.null(body) ||
    is.null(body$obj)
  ) {
    return(data.table())
  }

  resultado <- list()

  for (
    platform in names(TARGET_PLAYLIST_METRICS)
  ) {

    regs <- body$obj[[platform]]

    if (
      is.null(regs) ||
      length(regs) == 0 ||
      !is.list(regs)
    ) {
      next
    }

    metrics <- TARGET_PLAYLIST_METRICS[[platform]]
    filas_platform <- list()

    for (i in seq_along(regs)) {

      r <- regs[[i]]

      if (!is.list(r)) {
        next
      }

      if (!("created_at" %in% names(r))) {
        next
      }

      filas_metricas <- lapply(
        metrics,
        function(metric) {

          data.table(
            artist_id = as.character(artist_id),
            endpoint = "playlist_evolution",
            source = platform,
            metric = metric,
            timestamp = scalar_chr(r$created_at),
            value = scalar_num(r[[metric]]),
            interpolated = NA,
            extrapolated = NA,
            flags = NA_character_
          )
        }
      )

      filas_platform[[i]] <- rbindlist(
        filas_metricas,
        fill = TRUE
      )
    }

    filas_platform <- Filter(
      Negate(is.null),
      filas_platform
    )

    if (length(filas_platform) > 0) {
      resultado[[platform]] <- rbindlist(
        filas_platform,
        fill = TRUE
      )
    }
  }

  if (length(resultado) == 0) {
    return(data.table())
  }

  rbindlist(
    resultado,
    fill = TRUE
  )
}

In [ ]:
# ============================================================
# PLAYLIST EVOLUTION
# PROCESAMIENTO + EXTRACCIÓN
# Arquitectura individual por request
# Período explícito: FAN_SINCE a END_DATE
# ============================================================


playlist_raw_to_monthly <- function(raw) {

  if (
    is.null(raw) ||
    nrow(raw) == 0
  ) {
    return(
      data.table()
    )
  }


  d <- copy(
    as.data.table(raw)
  )


  d[
    ,
    timestamp_dt :=
      parse_timestamp(
        timestamp
      )
  ]


  d[
    ,
    observed_date :=
      as.Date(
        timestamp_dt
      )
  ]


  d <- d[
    !is.na(observed_date) &
      observed_date >= FAN_SINCE &
      observed_date <= END_DATE
  ]


  if (
    nrow(d) == 0
  ) {
    return(
      data.table()
    )
  }


  d[
    ,
    variable := paste(
      endpoint,
      source,
      metric,
      sep = "__"
    )
  ]


  d[
    ,
    month := as.Date(
      format(
        observed_date,
        "%Y-%m-01"
      )
    )
  ]


  setorder(
    d,
    artist_id,
    endpoint,
    source,
    metric,
    month,
    timestamp_dt
  )


  # Última observación disponible de cada mes.

  monthly <- d[
    ,
    .SD[.N],
    by = .(
      artist_id,
      endpoint,
      source,
      metric,
      variable,
      month
    )
  ]


  monthly[
    ,
    .(
      artist_id,
      month,
      endpoint,
      source,
      metric,
      variable,
      observed_date,
      value,
      interpolated,
      extrapolated,
      flags
    )
  ]
}


process_playlist_body <- function(
  body,
  artist_id,
  status_code_value = 200L,
  response_headers = list(),
  note = ""
) {

  request_key <- playlist_request_key(
    artist_id
  )


  platforms <- if (
    !is.null(body) &&
      !is.null(body$obj)
  ) {

    names(
      body$obj
    )

  } else {

    character()
  }


  raw <- parse_playlist_evolution(
    body,
    artist_id
  )


  monthly <- playlist_raw_to_monthly(
    raw
  )


  # Guardar el monthly antes de registrar
  # el request como completado.

  save_monthly_response(
    request_key,
    monthly
  )


  fechas <- if (
    nrow(raw) > 0
  ) {

    as.Date(
      parse_timestamp(
        raw$timestamp
      )
    )

  } else {

    as.Date(
      character()
    )
  }


  write_log_row(
    data.table(
      request_key =
        request_key,

      artist_id =
        as.character(
          artist_id
        ),

      endpoint =
        "playlist_evolution",

      source =
        "all",

      requested_since =
        format(
          FAN_SINCE,
          "%Y-%m-%d"
        ),

      requested_until =
        format(
          END_DATE,
          "%Y-%m-%d"
        ),

      status_code =
        as.character(
          status_code_value
        ),

      ok =
        "TRUE",

      returned_fields =
        paste(
          platforms,
          collapse = ";"
        ),

      n_rows =
        as.character(
          nrow(raw)
        ),

      min_date =
        safe_min_date(
          fechas
        ),

      max_date =
        safe_max_date(
          fechas
        ),

      rate_limit_limit =
        header_value(
          response_headers,
          "x-ratelimit-limit"
        ),

      rate_limit_remaining =
        header_value(
          response_headers,
          "x-ratelimit-remaining"
        ),

      rate_limit_reset =
        header_value(
          response_headers,
          "x-ratelimit-reset"
        ),

      completed_at =
        format(
          Sys.time(),
          "%Y-%m-%d %H:%M:%S"
        ),

      note =
        note
    )
  )


  cat(
    "  Filas objetivo parseadas:",
    nrow(raw),
    "\n"
  )


  cat(
    "  Filas mensuales guardadas:",
    nrow(monthly),
    "\n"
  )


  invisible(
    monthly
  )
}


extract_playlist_evolution <- function(
  artist_id,
  log_snapshot = NULL
) {

  artist_id <- as.character(
    artist_id
  )


  request_key <- playlist_request_key(
    artist_id
  )


  # Cuando se llama desde la cola, recibe un único
  # snapshot del log compartido por toda la ejecución.

  if (
    is.null(log_snapshot)
  ) {

    log_snapshot <- read_request_log()
  }


  completed_log <- if (
    nrow(log_snapshot) > 0
  ) {

    any(
      log_snapshot$request_key ==
        request_key &
        !is.na(log_snapshot$ok) &
        toupper(log_snapshot$ok) ==
          "TRUE"
    )

  } else {

    FALSE
  }


  monthly_cache <-
    monthly_response_exists(
      request_key
    )


  # ----------------------------------------------------------
  # 1. Monthly + log exitoso:
  #    request completamente terminado.
  # ----------------------------------------------------------

  if (
    completed_log &&
      monthly_cache
  ) {

    cat(
      "SKIP: playlist evolution histórico - ",
      "ya completado\n",
      sep = ""
    )

    return(
      invisible(NULL)
    )
  }


  # ----------------------------------------------------------
  # 2. Existe raw:
  #    reconstruir monthly o log localmente.
  # ----------------------------------------------------------

  if (
    raw_response_exists(
      request_key
    )
  ) {

    cat(
      "CACHE: playlist evolution histórico - artista ",
      artist_id,
      "\n",
      sep = ""
    )


    body <- get_raw_response(
      request_key
    )


    return(
      process_playlist_body(
        body = body,
        artist_id = artist_id,
        status_code_value = 200L,
        response_headers = list(),

        note =
          paste(
            "Procesado desde cache crudo local.",
            "Período:",
            format(
              FAN_SINCE,
              "%Y-%m-%d"
            ),
            "a",
            format(
              END_DATE,
              "%Y-%m-%d"
            )
          )
      )
    )
  }


  # ----------------------------------------------------------
  # 3. No existe raw:
  #    realizar el request histórico.
  # ----------------------------------------------------------

  cat(
    "REQUEST: playlist evolution histórico - artista ",
    artist_id,
    "\n",
    sep = ""
  )


  resp <- cm_get(
    path = paste0(
      "/api/playlist/by/artist/",
      artist_id,
      "/evolution"
    ),

    query = list(
      since =
        format(
          FAN_SINCE,
          "%Y-%m-%d"
        ),

      until =
        format(
          END_DATE,
          "%Y-%m-%d"
        )
    )
  )


  if (
    !resp$ok
  ) {

    write_log_row(
      data.table(
        request_key =
          request_key,

        artist_id =
          artist_id,

        endpoint =
          "playlist_evolution",

        source =
          "all",

        requested_since =
          format(
            FAN_SINCE,
            "%Y-%m-%d"
          ),

        requested_until =
          format(
            END_DATE,
            "%Y-%m-%d"
          ),

        status_code =
          as.character(
            resp$status
          ),

        ok =
          "FALSE",

        returned_fields =
          NA_character_,

        n_rows =
          "0",

        min_date =
          NA_character_,

        max_date =
          NA_character_,

        rate_limit_limit =
          header_value(
            resp$headers,
            "x-ratelimit-limit"
          ),

        rate_limit_remaining =
          header_value(
            resp$headers,
            "x-ratelimit-remaining"
          ),

        rate_limit_reset =
          header_value(
            resp$headers,
            "x-ratelimit-reset"
          ),

        completed_at =
          format(
            Sys.time(),
            "%Y-%m-%d %H:%M:%S"
          ),

        note =
          substr(
            resp$text,
            1,
            1000
          )
      )
    )


    cat(
      "  ERROR HTTP:",
      ifelse(
        is.na(
          resp$status
        ),
        "sin status",
        resp$status
      ),
      "\n"
    )


    return(
      invisible(NULL)
    )
  }


  # Preservar el raw antes de procesarlo.

  save_raw_response(
    request_key,
    resp$body
  )


  process_playlist_body(
    body = resp$body,
    artist_id = artist_id,
    status_code_value = resp$status,
    response_headers = resp$headers,

    note =
      paste(
        "Descargado desde API.",
        "Período solicitado:",
        format(
          FAN_SINCE,
          "%Y-%m-%d"
        ),
        "a",
        format(
          END_DATE,
          "%Y-%m-%d"
        )
      )
  )
}

## 9. Selección de la muestra por deciles de `cm_artist_rank`

La muestra creció por etapas (150, 300, 2000 y ampliación dentro del marco top 5000). Cada etapa conserva los artistas ya sorteados y usa semilla fija, así que la selección se reproduce al volver a correr.

In [ ]:
# ============================================================
# SELECCIÓN DE ARTISTAS PARA PILOTO 2 SEGÚN cm_artist_rank
# ============================================================

rank_data <- copy(ARTISTS)

# Asegurar que cm_artist_rank sea numérico
rank_data[
  ,
  cm_artist_rank := as.numeric(cm_artist_rank)
]

# Eliminar artistas sin rank válido
rank_data <- rank_data[
  !is.na(cm_artist_rank) &
  is.finite(cm_artist_rank)
]

cat(
  "Artistas con cm_artist_rank válido:",
  nrow(rank_data),
  "\n"
)

cat(
  "Rank mínimo:",
  min(rank_data$cm_artist_rank),
  "\n"
)

cat(
  "Rank máximo:",
  max(rank_data$cm_artist_rank),
  "\n"
)

cat(
  "Ranks únicos:",
  uniqueN(rank_data$cm_artist_rank),
  "\n"
)

# Quantiles empíricos de cm_artist_rank
quantiles_rank <- quantile(
  rank_data$cm_artist_rank,
  probs = seq(0, 1, 0.1),
  na.rm = TRUE,
  type = 7
)

quantiles_rank

In [ ]:
# Ordenar desde mejor rank hacia peor rank
setorder(
  rank_data,
  cm_artist_rank
)

# Construir 10 grupos con aproximadamente
# la misma cantidad de artistas
rank_data[
  ,
  decile := pmin(
    10L,
    ceiling(
      seq_len(.N) * 10 / .N
    )
  )
]

# Resumen de los deciles
decile_summary <- rank_data[
  ,
  .(
    n_artists = .N,
    rank_min = min(cm_artist_rank),
    rank_median = median(cm_artist_rank),
    rank_max = max(cm_artist_rank)
  ),
  by = decile
]

decile_summary

### 9.1 Muestra base de 150 artistas

In [ ]:
# ============================================================
# 150 artistas: 15 por decil de cm_artist_rank
#
# Conserva:
# - Taylor Swift (Piloto 1)
# - los 10 artistas originales del Piloto 2
#
# Completa cada decil mediante muestreo aleatorio
# con semilla fija.
# ============================================================

set.seed(12345)

# Taylor Swift
PILOT1_ARTIST_ID <- "2762"

# Los 10 artistas originales del Piloto 2
PILOT2_ORIGINAL_IDS <- c(
  "1399034",
  "4313",
  "21",
  "1360725",
  "1618903",
  "207878",
  "1038906",
  "1107040",
  "3489278",
  "104129"
)

# Artistas ya descargados
EXISTING_SAMPLE_IDS <- unique(
  c(
    PILOT1_ARTIST_ID,
    PILOT2_ORIGINAL_IDS
  )
)

# Asegurar ID como character
rank_data[
  ,
  ID := as.character(ID)
]

# Verificar cuántos de los ya descargados hay en cada decil
existing_by_decile <- rank_data[
  ID %in% EXISTING_SAMPLE_IDS,
  .(
    n_existing = uniqueN(ID)
  ),
  by = decile
]

# Completar deciles que eventualmente no aparezcan
existing_by_decile <- merge(
  data.table(
    decile = 1:10
  ),
  existing_by_decile,
  by = "decile",
  all.x = TRUE
)

existing_by_decile[
  is.na(n_existing),
  n_existing := 0L
]

# Cantidad de artistas nuevos necesarios por decil
existing_by_decile[
  ,
  n_to_sample := 15L - n_existing
]

existing_by_decile

In [ ]:
# ============================================================
# SELECCIONAR LOS ARTISTAS ADICIONALES
# Construcción de la muestra base de 150 artistas
# ============================================================

additional_selection <- rbindlist(
  lapply(
    1:10,
    function(d) {

      n_needed <- existing_by_decile[
        decile == d,
        n_to_sample
      ]

      candidates <- rank_data[
        decile == d &
          !ID %in% EXISTING_SAMPLE_IDS
      ]

      if (n_needed < 0) {
        stop(
          paste(
            "El decil",
            d,
            "ya tiene más de 15 artistas seleccionados."
          )
        )
      }

      if (nrow(candidates) < n_needed) {
        stop(
          paste(
            "No hay suficientes candidatos en el decil",
            d
          )
        )
      }

      if (n_needed == 0) {
        return(
          candidates[0]
        )
      }

      candidates[
        sample(
          .N,
          n_needed
        )
      ]
    }
  ),
  fill = TRUE
)


# ------------------------------------------------------------
# Muestra base final de 150 artistas
# ------------------------------------------------------------

PILOT2_ARTIST_IDS <- unique(
  c(
    EXISTING_SAMPLE_IDS,
    additional_selection$ID
  )
)

PILOT2_ARTIST_IDS <- as.character(
  PILOT2_ARTIST_IDS
)


cat(
  "Artistas ya descargados incluidos:",
  length(EXISTING_SAMPLE_IDS),
  "\n"
)

cat(
  "Artistas adicionales seleccionados:",
  nrow(additional_selection),
  "\n"
)

cat(
  "Total muestra Piloto 2:",
  length(PILOT2_ARTIST_IDS),
  "\n"
)

In [ ]:
# ============================================================
# VERIFICAR 15 ARTISTAS POR DECIL
# ============================================================

rank_data[
  ID %in% PILOT2_ARTIST_IDS,
  .(
    n_artists = uniqueN(ID)
  ),
  by = decile
][
  order(decile)
]

### 9.2 Ampliación a 300 artistas

In [ ]:
# ============================================================
# AMPLIACIÓN DEL PILOTO 2
# Base fija: los 150 artistas ya descargados
# ============================================================

BASE_150_IDS <- as.character(
  PILOT2_ARTIST_IDS
)

# Cantidad TOTAL deseada por decil.
# 30 por decil = 300 artistas.
TARGET_PER_DECILE <- 30L


# ------------------------------------------------------------
# Crear una única cola aleatoria de candidatos
# entre los artistas que todavía NO fueron usados.
# ------------------------------------------------------------

set.seed(20260909)

EXPANSION_POOL <- copy(
  rank_data[
    !ID %in% BASE_150_IDS
  ]
)

EXPANSION_POOL[
  ,
  random_order := sample.int(.N),
  by = decile
]

setorder(
  EXPANSION_POOL,
  decile,
  random_order
)


# ------------------------------------------------------------
# Seleccionar solamente los artistas faltantes
# para alcanzar TARGET_PER_DECILE
# ------------------------------------------------------------

PILOT2_NEW_IDS <- unlist(
  lapply(
    1:10,
    function(d) {

      n_existing <- rank_data[
        decile == d &
          ID %in% BASE_150_IDS,
        uniqueN(ID)
      ]

      n_needed <- TARGET_PER_DECILE -
        n_existing

      if (n_needed <= 0) {
        return(character())
      }

      head(
        EXPANSION_POOL[
          decile == d,
          ID
        ],
        n_needed
      )
    }
  ),
  use.names = FALSE
)


# ------------------------------------------------------------
# Muestra ampliada
# Los 150 viejos quedan primero y los nuevos después.
# ------------------------------------------------------------

PILOT2_ARTIST_IDS <- unique(
  c(
    BASE_150_IDS,
    PILOT2_NEW_IDS
  )
)


cat(
  "Artistas anteriores:",
  length(BASE_150_IDS),
  "\n"
)

cat(
  "Artistas nuevos:",
  length(PILOT2_NEW_IDS),
  "\n"
)

cat(
  "Total:",
  length(PILOT2_ARTIST_IDS),
  "\n\n"
)


# Verificación por decil
rank_data[
  ID %in% PILOT2_ARTIST_IDS,
  .(
    n_artists = uniqueN(ID)
  ),
  by = decile
][
  order(decile)
]

### 9.3 Ampliación a 2000 artistas (fija desde el 30/9/2026)

In [ ]:
# ============================================================
# NUEVA AMPLIACIÓN
# Base fija: los 300 artistas ya seleccionados
# ============================================================

BASE_300_IDS <- unique(
  as.character(
    PILOT2_ARTIST_IDS
  )
)

if (
  length(BASE_300_IDS) != 300
) {

  stop(
    "La base previa debería contener exactamente 300 artistas."
  )
}


# ------------------------------------------------------------
# FIJO EN 2000 DESDE EL 30/9/2026: es la muestra ya descargada y
# la base de la celda siguiente (AMPLIACIÓN DENTRO DEL MARCO TOP
# 5000), que es donde se amplía de ahora en más. No modificar.
# ------------------------------------------------------------

TARGET_TOTAL_ARTISTS <- 2000


if (
  TARGET_TOTAL_ARTISTS < 300
) {

  stop(
    "TARGET_TOTAL_ARTISTS no puede ser menor que 300."
  )
}


if (
  TARGET_TOTAL_ARTISTS %% 10 != 0
) {

  stop(
    "TARGET_TOTAL_ARTISTS debe ser múltiplo de 10."
  )
}


TARGET_PER_DECILE_NEXT <-
  TARGET_TOTAL_ARTISTS %/% 10


# ------------------------------------------------------------
# Orden permanente para todos los artistas posteriores a 300
# ------------------------------------------------------------

set.seed(
  20260910
)

EXPANSION_POOL_AFTER_300 <- copy(
  rank_data[
    !ID %in% BASE_300_IDS
  ]
)

EXPANSION_POOL_AFTER_300[
  ,
  random_order :=
    sample.int(.N),
  by = decile
]

setorder(
  EXPANSION_POOL_AFTER_300,
  decile,
  random_order
)


# ------------------------------------------------------------
# Artistas adicionales necesarios
# ------------------------------------------------------------

PILOT2_NEW_IDS <- unlist(
  lapply(
    1:10,
    function(d) {

      n_existing <- rank_data[
        decile == d &
          ID %in% BASE_300_IDS,
        uniqueN(ID)
      ]

      n_needed <-
        TARGET_PER_DECILE_NEXT -
        n_existing

      if (n_needed <= 0) {

        return(
          character()
        )
      }

      head(
        EXPANSION_POOL_AFTER_300[
          decile == d,
          ID
        ],
        n_needed
      )
    }
  ),
  use.names = FALSE
)


PILOT2_ARTIST_IDS <- unique(
  c(
    BASE_300_IDS,
    PILOT2_NEW_IDS
  )
)


cat(
  "Base previa:",
  length(BASE_300_IDS),
  "\n"
)

cat(
  "Artistas de expansión:",
  length(PILOT2_NEW_IDS),
  "\n"
)

cat(
  "Total objetivo:",
  length(PILOT2_ARTIST_IDS),
  "\n\n"
)


rank_data[
  ID %in% PILOT2_ARTIST_IDS,
  .(
    n_artists =
      uniqueN(ID)
  ),
  by = decile
][
  order(decile)
]

### 9.4 Ampliación dentro del marco top 5000

In [ ]:
# ============================================================
# AMPLIACIÓN DENTRO DEL MARCO TOP 5000
#
# Desde el 30/9/2026 la muestra crece sólo dentro del marco de
# muestreo: los MARCO_TOP_N artistas mejor rankeados del dataset
# maestro, con deciles recalculados en esa escala (decil_marco).
#
# La celda anterior queda fija en TARGET_TOTAL_ARTISTS = 2000:
# es la muestra ya descargada (sorteada por deciles de los 7000)
# y funciona como base. Acá se completa cada decil del marco
# hasta TARGET_POR_DECIL_MARCO artistas, con sorteo aleatorio y
# semilla fija dentro de cada decil. Subir el objetivo sólo
# agrega artistas: los ya sorteados se conservan.
# Los artistas de la base que quedan fuera del marco también se
# conservan: ya están descargados y sirven como contraste.
#
# Desde el 7/10/2026 la base incluye además los 99 artistas del
# marco recuperados por tramos (RECUPERACION_FAN_IDS). Salen del
# mismo sorteo por deciles de los 7000 que la base de 2000
# (llevar la muestra a 2140), así que siguen siendo una muestra
# aleatoria del marco. Se cuentan como ya incluidos y se sacan
# del pool después de asignar el orden aleatorio: el orden de
# los demás no cambia y los artistas ya sorteados se conservan.
#
# ÚNICO PARÁMETRO A CAMBIAR EN FUTURAS AMPLIACIONES:
# TARGET_POR_DECIL_MARCO (0 = sin ampliación). Con la base de
# 2000 más los 99 recuperados, el decil más cargado tiene 162
# artistas: a partir de ese valor todos los deciles quedan con
# la misma cantidad. Valores anteriores: 144, 148, 151 (hasta
# el 6/10/2026, sin contar los recuperados).
# Cada artista nuevo cuesta unos 31 requests (30 de Fan Stats
# por tramos y 1 de Playlist Evolution): unos 30 artistas por
# día de cuota. La celda informa el costo pendiente real.
# ============================================================

MARCO_TOP_N <- 5000L

TARGET_POR_DECIL_MARCO <- 176L


# ------------------------------------------------------------
# Marco: top N del dataset maestro por cm_artist_rank, con
# deciles propios. Se reutiliza en la sección 16.
# ------------------------------------------------------------

construir_marco_top <- function(rank_data, top_n) {

  marco <- copy(rank_data)[
    ,
    .(ID = as.character(ID), cm_artist_rank, decil_7000 = decile)
  ]

  setorder(marco, cm_artist_rank)

  marco[, posicion_rank := seq_len(.N)]

  marco <- marco[posicion_rank <= top_n]

  marco[
    ,
    decil_marco := as.integer(pmin(10L, ceiling(posicion_rank * 10 / top_n)))
  ]

  marco
}


# ------------------------------------------------------------
# Base: la muestra de 2000 de la celda anterior
# ------------------------------------------------------------


# La base se arma con las salidas fijas de la celda anterior, no con
# PILOT2_ARTIST_IDS, que esta celda reescribe: así se puede volver a
# correr después de cambiar TARGET_POR_DECIL_MARCO.
BASE_2000_IDS <- unique(as.character(c(BASE_300_IDS, PILOT2_NEW_IDS)))


if (length(BASE_2000_IDS) != 2000) {

  stop(
    "La base de la ampliación dentro del marco debería tener 2000 artistas. ",
    "Revisar TARGET_TOTAL_ARTISTS en la celda anterior (fijo en 2000)."
  )
}

marco_top_ampliacion <- construir_marco_top(rank_data, MARCO_TOP_N)


# ------------------------------------------------------------
# Recuperados: 99 artistas del marco con Fan Stats descargados
# por tramos (celda "FAN STATS: RECUPERACION UNICA DE 99
# ARTISTAS DEL MARCO", que usa esta misma lista)
# ------------------------------------------------------------

RECUPERACION_FAN_IDS <- as.character(
  c(
    63L, 72L, 319L, 507L, 642L, 786L, 801L, 1035L, 1065L, 1069L,
    1102L, 1243L, 1316L, 1843L, 1862L, 2005L, 2316L, 2510L, 2774L,
    2975L, 3132L, 3442L, 3474L, 3623L, 3854L, 4007L, 4139L, 4162L,
    4374L, 5102L, 5403L, 18854L, 18975L, 46553L, 50161L, 62878L,
    63444L, 78434L, 79226L, 94197L, 114674L, 130331L, 135886L,
    142624L, 174087L, 180047L, 185262L, 193790L, 195452L, 203331L,
    204492L, 206444L, 206585L, 206679L, 210443L, 210920L, 214957L,
    219658L, 231667L, 250194L, 253113L, 256873L, 280602L, 298790L,
    298917L, 299121L, 310930L, 418499L, 491650L, 505705L, 542632L,
    660193L, 717312L, 746940L, 748317L, 819190L, 824564L, 865697L,
    1142242L, 1151187L, 1232218L, 1366884L, 1402414L, 1410939L,
    1452714L, 1530639L, 1584047L, 1610625L, 1783398L, 3733653L,
    4407094L, 5074511L, 5589383L, 6347882L, 6618355L, 9354370L,
    9408465L, 9566342L, 13027884L
  )
)

stopifnot(
  "La lista de recuperados debe tener 99 artistas" =
    length(unique(RECUPERACION_FAN_IDS)) == 99,
  "Todos los recuperados deben estar en el marco" =
    all(RECUPERACION_FAN_IDS %in% marco_top_ampliacion$ID),
  "Ningún recuperado debe estar en la base de 2000" =
    !any(RECUPERACION_FAN_IDS %in% BASE_2000_IDS)
)


# ------------------------------------------------------------
# Artistas a sumar por decil del marco
# ------------------------------------------------------------

ampliacion_por_decil_marco <- marco_top_ampliacion[
  ,
  .(
    en_base = sum(ID %in% BASE_2000_IDS),
    recuperados = sum(ID %in% RECUPERACION_FAN_IDS)
  ),
  by = decil_marco
][order(decil_marco)]

ampliacion_por_decil_marco[
  ,
  a_sumar := pmax(0L, TARGET_POR_DECIL_MARCO - en_base - recuperados)
]


# ------------------------------------------------------------
# Orden permanente del pool: artistas del marco fuera de la
# base, en orden aleatorio dentro de cada decil del marco
# ------------------------------------------------------------

set.seed(
  20260930
)

POOL_MARCO_AMPLIACION <- copy(
  marco_top_ampliacion[!ID %in% BASE_2000_IDS]
)

POOL_MARCO_AMPLIACION[
  ,
  random_order := sample.int(.N),
  by = decil_marco
]

setorder(
  POOL_MARCO_AMPLIACION,
  decil_marco,
  random_order
)

# Artistas sorteados hasta el 6/10/2026 (objetivo 151 sin contar
# los recuperados), para controlar que sigan en la muestra
SORTEADOS_HASTA_151_IDS <- unlist(
  lapply(
    1:10,
    function(d) {
      head(
        POOL_MARCO_AMPLIACION[decil_marco == d, ID],
        max(0L, 151L - ampliacion_por_decil_marco[decil_marco == d, en_base])
      )
    }
  ),
  use.names = FALSE
)

# Los recuperados ya integran la muestra: salen del pool sin
# alterar el orden de los demás
POOL_MARCO_AMPLIACION <- POOL_MARCO_AMPLIACION[
  !ID %in% RECUPERACION_FAN_IDS
]

NUEVOS_MARCO_IDS <- unlist(
  lapply(
    1:10,
    function(d) {
      head(
        POOL_MARCO_AMPLIACION[decil_marco == d, ID],
        ampliacion_por_decil_marco[decil_marco == d, a_sumar]
      )
    }
  ),
  use.names = FALSE
)

PILOT2_ARTIST_IDS <- unique(
  c(
    BASE_2000_IDS,
    RECUPERACION_FAN_IDS,
    NUEVOS_MARCO_IDS
  )
)

if (!all(SORTEADOS_HASTA_151_IDS %in% PILOT2_ARTIST_IDS)) {

  warning(
    "Con este objetivo quedan fuera de la muestra artistas ya ",
    "sorteados y descargados: ",
    sum(!SORTEADOS_HASTA_151_IDS %in% PILOT2_ARTIST_IDS),
    ". Usar TARGET_POR_DECIL_MARCO de 162 o más."
  )
}

ampliacion_por_decil_marco[
  ,
  total := en_base + recuperados + a_sumar
]

# ------------------------------------------------------------
# Costo real: sólo las tareas que no figuran completas en el
# log (5 tramos por plataforma de Fan Stats y 1 request de
# Playlist Evolution). Los sorteados en ampliaciones anteriores
# ya están descargados y no cuestan requests.
# ------------------------------------------------------------

log_ok_ampliacion <- read_request_log()[toupper(ok) == "TRUE"]

fan_completas_ampliacion <- log_ok_ampliacion[
  endpoint == "fan_stats" &
    suppressWarnings(as.Date(requested_until)) >= END_DATE,
  unique(paste(artist_id, source))
]

pendientes_ampliacion <- CJ(
  artist_id = NUEVOS_MARCO_IDS,
  source = FAN_SOURCES
)[
  ,
  .(tramos = 5L * sum(!paste(artist_id, source) %in% fan_completas_ampliacion)),
  by = artist_id
][
  ,
  playlist := as.integer(
    !playlist_request_key(artist_id) %in% log_ok_ampliacion$request_key
  )
][
  ,
  requests := tramos + playlist
]

cat(
  "Base (celda anterior):", length(BASE_2000_IDS), "\n",
  "Recuperados del marco (Fan Stats por tramos):",
  length(RECUPERACION_FAN_IDS), "\n",
  "Objetivo por decil del marco:", TARGET_POR_DECIL_MARCO, "\n",
  "Artistas sorteados dentro del marco (incluye ya descargados):",
  length(NUEVOS_MARCO_IDS), "\n",
  "Total de la muestra:", length(PILOT2_ARTIST_IDS), "\n",
  "Artistas pendientes de descarga:",
  pendientes_ampliacion[requests > 0, .N], "\n",
  "Requests a la API pendientes:",
  pendientes_ampliacion[, sum(requests)], "\n\n"
)

ampliacion_por_decil_marco

## 10. Extracción de Fan Stats por tramos de hasta 365 días

Desde el 28/9/2026 Chartmetric rechaza con error 400 los pedidos de Fan Stats de más de 365 días. La extracción se hace por tramos desde 2022-01-01, con cache por tramo y controles de cuota (errores consecutivos y reserva diaria).

In [ ]:
# ============================================================
# FAN STATS POR TRAMOS: PARÁMETROS Y FUNCIONES
#
# Chartmetric limita las requests de Fan Stats a 365 días
# entre since y until (desde el 28/9/2026). Cada tarea
# artista-source se pide en tramos, se cachea cada tramo por
# separado y, cuando están todos, se combinan y se procesan
# con las funciones originales (parse_fan_stats,
# fan_raw_to_monthly). El mensual queda con la clave original
# de la tarea, así consolidate_monthly_requests() lo encuentra.
# ============================================================


# Inicio del período pedido por tramos. 2021 no se pide: queda
# fuera del dataset (la ventana del panel empieza en 2022-03).
FAN_SINCE_TRAMOS <- as.Date("2022-01-01")

# until - since de cada tramo. 364 deja margen frente al límite
# de 365 días de la API también en años bisiestos.
MAX_DIAS_TRAMO_FAN <- 364

# Orden de prioridad: primero las plataformas del panel.
FAN_SOURCES_PRIORIDAD <- c(
  "spotify",
  "youtube_channel",
  "instagram",
  "youtube_artist",
  "twitter",
  "tiktok"
)

# Controles de cuota
MAX_ERRORES_CONSECUTIVOS <- 3L
RESERVA_REQUESTS_DIARIOS <- 10L


stopifnot(
  setequal(FAN_SOURCES_PRIORIDAD, FAN_SOURCES)
)


# ------------------------------------------------------------
# Tramos del período
# ------------------------------------------------------------

fan_tramos_periodo <- function(
  since = FAN_SINCE_TRAMOS,
  until = END_DATE,
  max_dias = MAX_DIAS_TRAMO_FAN
) {

  inicios <- as.Date(character())
  fines <- as.Date(character())

  inicio <- since

  while (inicio <= until) {

    fin <- min(inicio + max_dias, until)

    inicios <- c(inicios, inicio)
    fines <- c(fines, fin)

    inicio <- fin + 1
  }

  data.table(
    tramo = seq_along(inicios),
    since = inicios,
    until = fines
  )
}


FAN_TRAMOS <- fan_tramos_periodo()


fan_tramo_request_key <- function(
  artist_id,
  source,
  since,
  until
) {

  paste(
    "fan_stats",
    as.character(artist_id),
    source,
    format(since, "%Y-%m-%d"),
    format(until, "%Y-%m-%d"),
    "tramo",
    sep = "|"
  )
}


# ------------------------------------------------------------
# Tarea completa: request OK que cubre desde FAN_SINCE_TRAMOS
# hasta END_DATE. Reconoce tanto las descargas anteriores
# (desde 2021) como las nuevas por tramos (desde 2022).
# ------------------------------------------------------------

fan_completed_tramos <- function(
  log,
  artist_id_value,
  source_value
) {

  if (nrow(log) == 0) {
    return(FALSE)
  }

  rows <- log[
    endpoint == "fan_stats" &
      artist_id == as.character(artist_id_value) &
      source == source_value &
      toupper(ok) == "TRUE"
  ]

  if (nrow(rows) == 0) {
    return(FALSE)
  }

  since_dates <- suppressWarnings(as.Date(rows$requested_since))
  until_dates <- suppressWarnings(as.Date(rows$requested_until))

  any(
    !is.na(since_dates) &
      !is.na(until_dates) &
      since_dates <= FAN_SINCE_TRAMOS &
      until_dates >= END_DATE
  )
}


# ------------------------------------------------------------
# Estado de la cuota durante una ejecución
# ------------------------------------------------------------

nuevo_estado_cuota <- function() {

  estado <- new.env()

  estado$requests_api <- 0L
  estado$remaining <- NA_real_

  estado
}


# ------------------------------------------------------------
# Un tramo: cache o API. Devuelve list(ok, body, status, texto).
# ------------------------------------------------------------

obtener_tramo_fan <- function(
  artist_id,
  source,
  since,
  until,
  estado_cuota
) {

  key <- fan_tramo_request_key(artist_id, source, since, until)

  if (raw_response_exists(key)) {

    return(
      list(
        ok = TRUE,
        body = get_raw_response(key),
        status = 200L,
        texto = "cache",
        desde_cache = TRUE
      )
    )
  }

  # Reserva de cuota: no pedir si la última respuesta informó
  # que quedan RESERVA_REQUESTS_DIARIOS o menos.
  if (
    !is.na(estado_cuota$remaining) &&
      estado_cuota$remaining <= RESERVA_REQUESTS_DIARIOS
  ) {

    stop(
      paste0(
        "CUOTA_RESERVA: quedan ", estado_cuota$remaining,
        " requests en la cuota diaria (reserva: ",
        RESERVA_REQUESTS_DIARIOS, ")."
      ),
      call. = FALSE
    )
  }

  resp <- cm_get(
    path = paste0("/api/artist/", artist_id, "/stat/", source),
    query = list(
      since = format(since),
      until = format(until)
    )
  )

  estado_cuota$requests_api <- estado_cuota$requests_api + 1L

  remaining <- suppressWarnings(
    as.numeric(header_value(resp$headers, "x-ratelimit-remaining"))
  )

  if (!is.na(remaining)) {
    estado_cuota$remaining <- remaining
  }

  n_filas <- if (resp$ok) {
    nrow(parse_fan_stats(resp$body, artist_id, source))
  } else {
    0L
  }

  write_log_row(
    data.table(
      request_key = key,
      artist_id = as.character(artist_id),
      endpoint = "fan_stats_tramo",
      source = source,
      requested_since = format(since),
      requested_until = format(until),
      status_code = as.character(resp$status),
      ok = if (resp$ok) "TRUE" else "FALSE",
      returned_fields = if (resp$ok && !is.null(resp$body$obj)) {
        paste(names(resp$body$obj), collapse = ";")
      } else {
        NA_character_
      },
      n_rows = as.character(n_filas),
      min_date = NA_character_,
      max_date = NA_character_,
      rate_limit_limit = header_value(resp$headers, "x-ratelimit-limit"),
      rate_limit_remaining = header_value(resp$headers, "x-ratelimit-remaining"),
      rate_limit_reset = header_value(resp$headers, "x-ratelimit-reset"),
      completed_at = format(Sys.time(), "%Y-%m-%d %H:%M:%S"),
      note = if (resp$ok) {
        "Tramo descargado desde API"
      } else {
        substr(resp$text, 1, 1000)
      }
    )
  )

  if (resp$ok) {
    save_raw_response(key, resp$body)
  }

  list(
    ok = resp$ok,
    body = resp$body,
    status = resp$status,
    texto = if (resp$ok) "" else substr(resp$text, 1, 300),
    desde_cache = FALSE
  )
}


# ------------------------------------------------------------
# Combinar los tramos y guardar el mensual con la clave
# original de la tarea (fan_request_key)
# ------------------------------------------------------------

procesar_tramos_fan <- function(
  bodies,
  artist_id,
  source,
  n_desde_api
) {

  request_key <- fan_request_key(artist_id, source)

  raw <- rbindlist(
    lapply(
      bodies,
      function(body) parse_fan_stats(body, artist_id, source)
    ),
    fill = TRUE
  )

  if (nrow(raw) > 0) {
    raw <- unique(raw, by = c("metric", "timestamp"))
  }

  monthly <- fan_raw_to_monthly(raw)

  save_monthly_response(request_key, monthly)

  fields <- unique(
    unlist(
      lapply(
        bodies,
        function(body) if (!is.null(body$obj)) names(body$obj) else character()
      )
    )
  )

  fechas <- if (nrow(raw) > 0) {
    as.Date(parse_timestamp(raw$timestamp))
  } else {
    as.Date(character())
  }

  write_log_row(
    data.table(
      request_key = request_key,
      artist_id = as.character(artist_id),
      endpoint = "fan_stats",
      source = source,
      requested_since = format(FAN_SINCE_TRAMOS),
      requested_until = format(END_DATE),
      status_code = "200",
      ok = "TRUE",
      returned_fields = paste(fields, collapse = ";"),
      n_rows = as.character(nrow(raw)),
      min_date = safe_min_date(fechas),
      max_date = safe_max_date(fechas),
      rate_limit_limit = NA_character_,
      rate_limit_remaining = NA_character_,
      rate_limit_reset = NA_character_,
      completed_at = format(Sys.time(), "%Y-%m-%d %H:%M:%S"),
      note = paste0(
        "Combinado de ", length(bodies), " tramos de hasta 365 días (",
        format(FAN_SINCE_TRAMOS), " a ", format(END_DATE), "); ",
        n_desde_api, " tramos pedidos a la API en esta ejecución."
      )
    )
  )

  cat("  Filas objetivo parseadas:", nrow(raw), "\n")
  cat("  Filas mensuales guardadas:", nrow(monthly), "\n")

  invisible(monthly)
}


# ------------------------------------------------------------
# Una tarea artista-source.
# Devuelve list(estado, status, texto):
#   estado = "OK", "SKIP" o "ERROR".
# ------------------------------------------------------------

extract_fan_source_tramos <- function(
  artist_id,
  source,
  log_snapshot,
  estado_cuota
) {

  artist_id <- as.character(artist_id)

  request_key <- fan_request_key(artist_id, source)

  # 1. Tarea ya completa según el log: nunca vuelve a la API.
  #    Si le falta el mensual propio y hay cache crudo (completo
  #    o los tramos), se reconstruye sin requests. Si no hay
  #    cache (tareas legacy, guardadas directo en la base
  #    mensual consolidada), se saltea, igual que el código
  #    original.
  if (fan_completed_tramos(log_snapshot, artist_id, source)) {

    if (!monthly_response_exists(request_key)) {

      if (raw_response_exists(request_key)) {

        cat("CACHE:", source, "- reconstruyendo mensual\n")

        process_fan_body(
          body = get_raw_response(request_key),
          artist_id = artist_id,
          source = source,
          status_code_value = 200L,
          response_headers = list(),
          note = "Reconstruido desde cache crudo"
        )

        return(list(estado = "OK", status = 200L, texto = ""))
      }

      claves_tramos <- fan_tramo_request_key(
        artist_id, source, FAN_TRAMOS$since, FAN_TRAMOS$until
      )

      if (all(file.exists(raw_response_path(claves_tramos)))) {

        cat("CACHE:", source, "- reconstruyendo mensual desde tramos\n")

        procesar_tramos_fan(
          lapply(claves_tramos, get_raw_response),
          artist_id,
          source,
          n_desde_api = 0L
        )

        return(list(estado = "OK", status = 200L, texto = ""))
      }
    }

    cat("SKIP:", source, "- período ya cubierto\n")

    return(list(estado = "SKIP", status = NA_integer_, texto = ""))
  }

  # 2. Respuesta cruda completa anterior (período desde 2021)
  #    sin procesar: se procesa con la función original.
  if (raw_response_exists(request_key)) {

    cat("CACHE:", source, "- artista", artist_id, "\n")

    process_fan_body(
      body = get_raw_response(request_key),
      artist_id = artist_id,
      source = source,
      status_code_value = 200L,
      response_headers = list(),
      note = "Procesado desde cache crudo local"
    )

    return(list(estado = "OK", status = 200L, texto = ""))
  }

  # 3. Tramos: cache o API, en orden. Ante el primer tramo que
  #    falla, se corta la tarea sin pedir los siguientes.
  cat("TRAMOS:", source, "- artista", artist_id, "\n")

  bodies <- list()
  n_desde_api <- 0L

  for (i in seq_len(nrow(FAN_TRAMOS))) {

    tramo <- obtener_tramo_fan(
      artist_id,
      source,
      FAN_TRAMOS$since[i],
      FAN_TRAMOS$until[i],
      estado_cuota
    )

    if (!tramo$ok) {

      cat(
        "  ERROR HTTP en tramo ", i, " (",
        format(FAN_TRAMOS$since[i]), " a ", format(FAN_TRAMOS$until[i]), "): ",
        ifelse(is.na(tramo$status), "sin status", tramo$status), "\n",
        sep = ""
      )

      return(list(estado = "ERROR", status = tramo$status, texto = tramo$texto))
    }

    if (!tramo$desde_cache) {
      n_desde_api <- n_desde_api + 1L
    }

    bodies[[i]] <- tramo$body
  }

  procesar_tramos_fan(bodies, artist_id, source, n_desde_api)

  list(estado = "OK", status = 200L, texto = "")
}


# ------------------------------------------------------------
# Cola de trabajo e inventario
# ------------------------------------------------------------

build_fan_work_queue_tramos <- function(ids) {

  ids <- unique(as.character(ids))

  log_snapshot <- read_request_log()

  status <- CJ(
    source = FAN_SOURCES_PRIORIDAD,
    artist_id = ids,
    sorted = FALSE
  )

  # Tareas completas según el log, leído una sola vez (misma
  # regla que fan_completed_tramos)
  tareas_completas <- log_snapshot[
    endpoint == "fan_stats" & toupper(ok) == "TRUE"
  ][
    suppressWarnings(as.Date(requested_since)) <= FAN_SINCE_TRAMOS &
      suppressWarnings(as.Date(requested_until)) >= END_DATE,
    unique(paste(artist_id, source))
  ]

  status[
    ,
    completed := paste(artist_id, source) %in% tareas_completas
  ]

  status[
    ,
    `:=`(
      monthly_cache = file.exists(
        monthly_response_path(fan_request_key(artist_id, source))
      ),
      raw_cache = file.exists(
        raw_response_path(fan_request_key(artist_id, source))
      )
    )
  ]

  # Tramos ya cacheados, sólo para las tareas sin mensual propio
  status[, tramos_en_cache := 0L]

  status[
    monthly_cache == FALSE,
    tramos_en_cache := Reduce(
      `+`,
      lapply(
        seq_len(nrow(FAN_TRAMOS)),
        function(i) {
          as.integer(
            file.exists(
              raw_response_path(
                fan_tramo_request_key(
                  artist_id, source, FAN_TRAMOS$since[i], FAN_TRAMOS$until[i]
                )
              )
            )
          )
        }
      )
    )
  ]

  status[
    ,
    prioridad := match(source, FAN_SOURCES_PRIORIDAD)
  ]

  status[
    ,
    orden_artista := match(artist_id, ids)
  ]

  setorder(status, prioridad, orden_artista)

  # Misma regla que el código original: una tarea completa en el
  # log sólo tiene trabajo pendiente si le falta el mensual y hay
  # cache crudo para reconstruirlo (sin requests). Las tareas
  # legacy completas sin cache quedan fuera de la cola.
  status[
    ,
    needs_work := !completed |
      (
        !monthly_cache &
          (raw_cache | tramos_en_cache == nrow(FAN_TRAMOS))
      )
  ]

  status[
    ,
    requests_api_estimados := fifelse(
      !completed & !raw_cache,
      nrow(FAN_TRAMOS) - tramos_en_cache,
      0L
    )
  ]

  status
}


fan_request_inventory_tramos <- function(ids) {

  status <- build_fan_work_queue_tramos(ids)

  resumen <- status[
    ,
    .(
      tareas = .N,
      completas = sum(completed & !needs_work),
      pendientes = sum(needs_work),
      requests_api_estimados = sum(requests_api_estimados)
    ),
    by = source
  ]

  rbindlist(
    list(
      resumen,
      resumen[
        ,
        .(
          source = "TOTAL",
          tareas = sum(tareas),
          completas = sum(completas),
          pendientes = sum(pendientes),
          requests_api_estimados = sum(requests_api_estimados)
        )
      ]
    )
  )
}


cat("Tramos por tarea:\n")
print(FAN_TRAMOS)

In [ ]:
if (EJECUTAR_EXTRACCION_API) {

  # ============================================================
  # FAN STATS POR TRAMOS: EJECUTAR ÚNICAMENTE TAREAS PENDIENTES
  #
  # Cada ejecución reconstruye la cola a partir del log y del
  # cache (tareas completas, mensuales y tramos ya descargados).
  # Orden: plataformas del panel primero (FAN_SOURCES_PRIORIDAD),
  # y dentro de cada plataforma, el orden de PILOT2_ARTIST_IDS.
  #
  # La extracción se detiene si:
  # - MAX_ERRORES_CONSECUTIVOS tareas seguidas terminan en error;
  # - la cuota diaria informada baja a RESERVA_REQUESTS_DIARIOS;
  # - Chartmetric devuelve un rate limit largo (cm_get).
  # ============================================================

  fan_work_queue_tramos <- build_fan_work_queue_tramos(
    PILOT2_ARTIST_IDS
  )[
    needs_work == TRUE
  ]

  cat(
    "Tareas pendientes:", nrow(fan_work_queue_tramos), "\n",
    "Requests a la API estimados:",
    sum(fan_work_queue_tramos$requests_api_estimados), "\n\n"
  )

  print(
    fan_work_queue_tramos[
      ,
      .(tareas = .N, requests_api_estimados = sum(requests_api_estimados)),
      by = source
    ]
  )


  log_snapshot_tramos <- read_request_log()

  estado_cuota_tramos <- nuevo_estado_cuota()

  run_start_tramos <- Sys.time()

  errores_consecutivos <- 0L
  errores_recientes <- character()
  motivo_detencion <- "cola completa"


  if (nrow(fan_work_queue_tramos) > 0) {

    for (i in seq_len(nrow(fan_work_queue_tramos))) {

      id <- fan_work_queue_tramos$artist_id[i]
      source <- fan_work_queue_tramos$source[i]

      cat(
        "\n[", i, "/", nrow(fan_work_queue_tramos), "] ARTISTA ", id,
        " - ", source, "\n",
        sep = ""
      )

      flush.console()

      resultado <- tryCatch(
        extract_fan_source_tramos(
          id,
          source,
          log_snapshot = log_snapshot_tramos,
          estado_cuota = estado_cuota_tramos
        ),
        error = function(e) {

          mensaje <- conditionMessage(e)

          if (grepl("^(RATE_LIMIT_|CUOTA_RESERVA)", mensaje)) {
            return(list(estado = "DETENER", texto = mensaje))
          }

          stop(e)
        }
      )

      if (identical(resultado$estado, "DETENER")) {

        motivo_detencion <- resultado$texto

        cat("\n", resultado$texto, "\n", sep = "")

        break
      }

      if (identical(resultado$estado, "ERROR")) {

        errores_consecutivos <- errores_consecutivos + 1L

        errores_recientes <- c(
          errores_recientes,
          paste0(
            id, " - ", source, ": HTTP ",
            ifelse(is.na(resultado$status), "sin status", resultado$status),
            " ", resultado$texto
          )
        )

        if (errores_consecutivos >= MAX_ERRORES_CONSECUTIVOS) {

          motivo_detencion <- paste0(
            "ERRORES_CONSECUTIVOS: ", errores_consecutivos,
            " tareas seguidas terminaron en error."
          )

          cat("\n", motivo_detencion, "\n", sep = "")
          cat(paste0("  ", tail(errores_recientes, MAX_ERRORES_CONSECUTIVOS), collapse = "\n"), "\n")

          break
        }

      } else {

        errores_consecutivos <- 0L
      }
    }
  }


  # ------------------------------------------------------------
  # Resumen de esta ejecución
  # ------------------------------------------------------------

  cat(
    "\nMotivo de fin:", motivo_detencion, "\n",
    "Requests a la API en esta ejecución:", estado_cuota_tramos$requests_api, "\n",
    "Cuota restante informada por la API:",
    ifelse(is.na(estado_cuota_tramos$remaining), "sin dato", estado_cuota_tramos$remaining), "\n",
    "Tiempo de esta ejecución:",
    round(as.numeric(difftime(Sys.time(), run_start_tramos, units = "mins")), 1),
    "minutos\n\n"
  )

  cat("Estado actualizado:\n")

  fan_request_inventory_tramos(PILOT2_ARTIST_IDS)

} else {

  cat("Extraccion desactivada (EJECUTAR_EXTRACCION_API = FALSE): sin requests a la API.\n")
}

### 10.1 Recuperación de 99 artistas del marco

Artistas del marco que recibieron sólo errores 400 el 28 y el 29/9/2026. La descarga terminó el 6/10/2026 y desde el 7/10 integran la muestra.

In [ ]:
if (EJECUTAR_EXTRACCION_API) {

  # ============================================================
  # FAN STATS: RECUPERACION UNICA DE 99 ARTISTAS DEL MARCO
  #
  # Lista fija: artistas con error 400 en todas sus requests de
  # Fan Stats (28 y 29/9/2026), ninguna descarga OK de Fan Stats,
  # Playlist Evolution descargado y posicion <= 5000 en el marco
  # (cm_artist_rank). La lista se define en la celda AMPLIACION
  # DENTRO DEL MARCO TOP 5000: desde el 7/10/2026 estos artistas
  # integran la muestra (PILOT2_ARTIST_IDS).
  #
  # Costo: 5 requests por artista y plataforma (495 por plataforma,
  # 2970 en total, unos 3 dias de cuota). Spotify se completa
  # primero para todos los artistas.
  # Cada ejecucion retoma desde el log y el cache; con todo
  # completo, la cola queda vacia y no se hacen requests.
  #
  # REMANENTE SIN FAN STATS (registro, no se descarga):
  #
  # a) 39 de los 138, fuera del marco top 5000 (posiciones 5021 a 6962):
  #   1007L, 1375L, 3234L, 4881L, 30742L, 31036L, 53905L, 74811L,
  #   76319L, 95338L, 122178L, 168243L, 173817L, 179748L, 208633L,
  #   209753L, 210720L, 215428L, 256772L, 260852L, 410967L, 495072L,
  #   551004L, 576602L, 714587L, 775905L, 1020214L, 1355133L,
  #   1414746L, 3522548L, 3753800L, 3884445L, 3946757L, 3958657L,
  #   3972273L, 5294386L, 7626180L, 8046066L, 8092346L
  #
  # b) 80 artistas del marco pedidos el 29/9/2026 entre 17:26 y
  #    17:31 con la extraccion de periodo completo: solo errores
  #    400 y ningun archivo, tampoco de Playlist Evolution:
  #   69L, 133L, 166L, 190L, 277L, 377L, 758L, 1058L, 1279L, 1341L,
  #   1823L, 2101L, 2385L, 2488L, 2845L, 3295L, 3417L, 3530L, 3610L,
  #   3696L, 3826L, 3970L, 4014L, 4031L, 4147L, 4227L, 4515L, 4548L,
  #   4556L, 4642L, 5291L, 39691L, 61039L, 112095L, 138106L, 145086L,
  #   169478L, 182773L, 205962L, 206297L, 206565L, 207401L, 207545L,
  #   208141L, 208883L, 210333L, 214045L, 217671L, 224574L, 227995L,
  #   309041L, 310896L, 313672L, 373643L, 401355L, 450003L, 514689L,
  #   562618L, 601671L, 680402L, 718668L, 747993L, 839545L, 854911L,
  #   954776L, 1091042L, 1345095L, 1679272L, 3364617L, 3540881L,
  #   3571802L, 3646949L, 3654834L, 3946893L, 4158718L, 4175243L,
  #   5576162L, 5587268L, 5693620L, 11769965L
  # ============================================================

  stopifnot(
    "Falta correr la celda AMPLIACIÓN DENTRO DEL MARCO TOP 5000" =
      exists("RECUPERACION_FAN_IDS")
  )

  # Plataformas a recuperar. c("spotify") limita la descarga a
  # Spotify (495 requests en total).
  FUENTES_RECUPERACION <- FAN_SOURCES_PRIORIDAD


  stopifnot(
    "La lista de recuperacion debe tener 99 artistas" =
      length(unique(RECUPERACION_FAN_IDS)) == 99
  )

  cat(
    "Artistas de la recuperacion que ya integran la muestra:",
    sum(RECUPERACION_FAN_IDS %in% PILOT2_ARTIST_IDS), "
  "
  )


  cola_recuperacion_fan <- build_fan_work_queue_tramos(
    RECUPERACION_FAN_IDS
  )[
    needs_work == TRUE & source %in% FUENTES_RECUPERACION
  ]

  cat(
    "Tareas pendientes de recuperacion:", nrow(cola_recuperacion_fan), "\n",
    "Requests a la API estimados:",
    sum(cola_recuperacion_fan$requests_api_estimados), "\n\n"
  )


  if (nrow(cola_recuperacion_fan) == 0) {

    cat("Recuperacion completa: no quedan tareas pendientes. No se hacen requests.\n")

  } else {

    log_snapshot_recuperacion <- read_request_log()
    estado_cuota_recuperacion <- nuevo_estado_cuota()
    inicio_recuperacion <- Sys.time()

    errores_consecutivos_recuperacion <- 0L
    errores_recientes_recuperacion <- character()
    motivo_fin_recuperacion <- "cola completa"

    for (i in seq_len(nrow(cola_recuperacion_fan))) {

      id <- cola_recuperacion_fan$artist_id[i]
      source <- cola_recuperacion_fan$source[i]

      cat("\n[", i, "/", nrow(cola_recuperacion_fan), "] ARTISTA ", id, " - ", source, "\n", sep = "")
      flush.console()

      resultado <- tryCatch(
        extract_fan_source_tramos(
          id,
          source,
          log_snapshot = log_snapshot_recuperacion,
          estado_cuota = estado_cuota_recuperacion
        ),
        error = function(e) {
          mensaje <- conditionMessage(e)
          if (grepl("^(RATE_LIMIT_|CUOTA_RESERVA)", mensaje)) {
            return(list(estado = "DETENER", texto = mensaje))
          }
          stop(e)
        }
      )

      if (identical(resultado$estado, "DETENER")) {
        motivo_fin_recuperacion <- resultado$texto
        cat("\n", resultado$texto, "\n", sep = "")
        break
      }

      if (identical(resultado$estado, "ERROR")) {

        errores_consecutivos_recuperacion <- errores_consecutivos_recuperacion + 1L

        errores_recientes_recuperacion <- c(
          errores_recientes_recuperacion,
          paste0(id, " - ", source, ": HTTP ",
                 ifelse(is.na(resultado$status), "sin status", resultado$status),
                 " ", resultado$texto)
        )

        if (errores_consecutivos_recuperacion >= MAX_ERRORES_CONSECUTIVOS) {
          motivo_fin_recuperacion <- paste0(
            "ERRORES_CONSECUTIVOS: ", errores_consecutivos_recuperacion,
            " tareas seguidas terminaron en error."
          )
          cat("\n", motivo_fin_recuperacion, "\n", sep = "")
          cat(paste0("  ", tail(errores_recientes_recuperacion, MAX_ERRORES_CONSECUTIVOS), collapse = "\n"), "\n")
          break
        }

      } else {

        errores_consecutivos_recuperacion <- 0L
      }
    }

    cat(
      "\nMotivo de fin:", motivo_fin_recuperacion, "\n",
      "Requests a la API en esta ejecucion:", estado_cuota_recuperacion$requests_api, "\n",
      "Cuota restante informada por la API:",
      ifelse(is.na(estado_cuota_recuperacion$remaining), "sin dato", estado_cuota_recuperacion$remaining), "\n",
      "Tiempo de esta ejecucion:",
      round(as.numeric(difftime(Sys.time(), inicio_recuperacion, units = "mins")), 1), "minutos\n\n"
    )
  }


  # ------------------------------------------------------------
  # Estado de la recuperacion por plataforma
  # ------------------------------------------------------------

  fan_request_inventory_tramos(RECUPERACION_FAN_IDS)

} else {

  cat("Extraccion desactivada (EJECUTAR_EXTRACCION_API = FALSE): sin requests a la API.\n")
}

## 11. Extracción de Playlist Evolution

In [ ]:
# ============================================================
# PLAYLIST EVOLUTION
# INVENTARIO + COLA DINÁMICA
#
# - Lee el log una sola vez al comenzar.
# - Prioriza reprocesos locales.
# - Ejecuta únicamente artistas pendientes.
# - Lee nuevamente el log al finalizar.
# ============================================================


playlist_request_status <- function(
  ids,
  log_snapshot = NULL
) {

  ids <- unique(
    as.character(ids)
  )


  if (
    is.null(log_snapshot)
  ) {

    log_snapshot <- read_request_log()
  }


  completed_keys <- if (
    nrow(log_snapshot) > 0
  ) {

    unique(
      log_snapshot[
        !is.na(ok) &
          toupper(ok) == "TRUE",
        request_key
      ]
    )

  } else {

    character()
  }


  request_keys <- vapply(
    ids,
    playlist_request_key,
    FUN.VALUE = character(1),
    USE.NAMES = FALSE
  )


  raw_paths <- vapply(
    request_keys,
    raw_response_path,
    FUN.VALUE = character(1),
    USE.NAMES = FALSE
  )


  monthly_paths <- vapply(
    request_keys,
    monthly_response_path,
    FUN.VALUE = character(1),
    USE.NAMES = FALSE
  )


  completed_log <-
    request_keys %chin%
      completed_keys


  raw_cache <-
    file.exists(
      raw_paths
    )


  monthly_cache <-
    file.exists(
      monthly_paths
    )


  # Resultado analítico completo:
  # monthly disponible + log exitoso.

  complete <-
    monthly_cache &
    completed_log


  # Si existe raw, cualquier monthly o log faltante
  # puede reconstruirse localmente.

  needs_local <-
    raw_cache &
    !complete


  # Si no existe raw y el resultado tampoco está
  # completo, se necesita nuevamente la API.

  needs_api <-
    !raw_cache &
    !complete


  state <- fcase(
    complete,
    "complete",

    needs_local,
    "needs_local",

    needs_api,
    "needs_api",

    default =
      "review"
  )


  data.table(
    artist_id =
      ids,

    request_key =
      request_keys,

    completed_log =
      completed_log,

    raw_cache =
      raw_cache,

    monthly_cache =
      monthly_cache,

    complete =
      complete,

    needs_api =
      needs_api,

    needs_local =
      needs_local,

    state =
      state
  )
}


playlist_request_inventory <- function(
  ids,
  status = NULL
) {

  if (
    is.null(status)
  ) {

    status <- playlist_request_status(
      ids
    )
  }


  status[
    ,
    .(
      artistas =
        .N,

      completados_en_log =
        sum(
          completed_log
        ),

      completos_validos =
        sum(
          complete
        ),

      con_cache_crudo_nuevo =
        sum(
          raw_cache
        ),

      con_mensual_nuevo =
        sum(
          monthly_cache
        ),

      completos_sin_raw =
        sum(
          complete &
            !raw_cache
        ),

      requests_api_necesarios =
        sum(
          needs_api
        ),

      reprocesos_locales =
        sum(
          needs_local
        ),

      estados_para_revisar =
        sum(
          state == "review"
        )
    )
  ]
}


playlist_work_queue <- function(
  ids,
  status = NULL
) {

  if (
    is.null(status)
  ) {

    status <- playlist_request_status(
      ids
    )
  }


  queue <- status[
    needs_api |
      needs_local,
    .(
      artist_id,
      needs_api,
      needs_local,
      state
    )
  ]


  # Primero resolver los reprocesos locales.
  # No consumen requests de la API.

  queue[
    ,
    priority := fifelse(
      needs_local,
      1L,
      2L
    )
  ]


  setorder(
    queue,
    priority
  )


  queue[
    ,
    priority := NULL
  ]


  queue
}


# ============================================================
# CONSTRUIR INVENTARIO Y COLA
# Un único snapshot del log para toda esta ejecución.
# ============================================================

In [ ]:
if (EJECUTAR_EXTRACCION_API) {

  playlist_log_snapshot <-
    read_request_log()


  playlist_status_before <-
    playlist_request_status(
      PILOT2_ARTIST_IDS,
      log_snapshot =
        playlist_log_snapshot
    )


  cat(
    "Estado Playlist antes de comenzar:\n"
  )


  print(
    playlist_request_inventory(
      PILOT2_ARTIST_IDS,
      status =
        playlist_status_before
    )
  )


  playlist_queue <-
    playlist_work_queue(
      PILOT2_ARTIST_IDS,
      status =
        playlist_status_before
    )


  cat(
    "\nTareas Playlist pendientes:",
    nrow(playlist_queue),
    "\n"
  )


  cat(
    "Requieren API:",
    sum(
      playlist_queue$needs_api
    ),
    "\n"
  )


  cat(
    "Reprocesos locales:",
    sum(
      playlist_queue$needs_local
    ),
    "\n\n"
  )


  # ============================================================
  # EJECUTAR SOLAMENTE LA COLA PENDIENTE
  # ============================================================

  run_start <- Sys.time()

  rate_limit_hit <- FALSE


  if (
    nrow(playlist_queue) > 0
  ) {

    for (
      i in seq_len(
        nrow(playlist_queue)
      )
    ) {

      id <-
        playlist_queue$artist_id[i]


      cat(
        "\n[",
        i,
        "/",
        nrow(playlist_queue),
        "] ARTISTA ",
        id,
        " - ",
        playlist_queue$state[i],
        "\n",
        sep = ""
      )


      flush.console()


      resultado <- tryCatch(

        {

          extract_playlist_evolution(
            artist_id = id,
            log_snapshot =
              playlist_log_snapshot
          )

          "OK"
        },

        error = function(e) {

          mensaje <-
            conditionMessage(
              e
            )


          if (
            grepl(
              "^RATE_LIMIT_",
              mensaje
            )
          ) {

            cat(
              "\n",
              mensaje,
              "\n",
              sep = ""
            )

            return(
              "RATE_LIMIT"
            )
          }


          stop(
            e
          )
        }
      )


      if (
        identical(
          resultado,
          "RATE_LIMIT"
        )
      ) {

        rate_limit_hit <- TRUE


        cat(
          "\nExtracción Playlist pausada por rate limit.\n"
        )


        break
      }
    }
  }


  # ============================================================
  # RESUMEN DE LA EJECUCIÓN
  # ============================================================

  run_elapsed <- as.numeric(
    difftime(
      Sys.time(),
      run_start,
      units = "mins"
    )
  )


  cat(
    "\nTiempo de esta ejecución:",
    round(
      run_elapsed,
      1
    ),
    "minutos\n"
  )


  cat(
    "Rate limit detectado:",
    rate_limit_hit,
    "\n\n"
  )


  # Volver a leer el log una única vez después de la ejecución.

  playlist_log_after <-
    read_request_log()


  playlist_status_after <-
    playlist_request_status(
      PILOT2_ARTIST_IDS,
      log_snapshot =
        playlist_log_after
    )


  cat(
    "Estado Playlist actualizado:\n"
  )

} else {

  cat("Extraccion desactivada (EJECUTAR_EXTRACCION_API = FALSE): sin requests a la API.\n")
}

In [ ]:
playlist_request_inventory(
  PILOT2_ARTIST_IDS
)

## 12. Mensualización: último valor disponible del mes

In [ ]:
required_variables <- function() {

  fan <- unlist(
    lapply(
      names(TARGET_FAN_METRICS),
      function(source) {
        paste(
          "fan_stats",
          source,
          TARGET_FAN_METRICS[[source]],
          sep = "__"
        )
      }
    ),
    use.names = FALSE
  )

  playlist <- unlist(
    lapply(
      names(TARGET_PLAYLIST_METRICS),
      function(source) {
        paste(
          "playlist_evolution",
          source,
          TARGET_PLAYLIST_METRICS[[source]],
          sep = "__"
        )
      }
    ),
    use.names = FALSE
  )

  c(
    fan,
    playlist
  )
}

REQUIRED_VARIABLES <- required_variables()

# Subconjuntos por familia, usados en todo lo que sigue
# (cobertura por variable, cobertura conjunta, comparación de ventanas).
FAN_REQUIRED_VARIABLES <- REQUIRED_VARIABLES[
  grepl("^fan_stats__", REQUIRED_VARIABLES)
]

PLAYLIST_REQUIRED_VARIABLES <- REQUIRED_VARIABLES[
  grepl("^playlist_evolution__", REQUIRED_VARIABLES)
]

REQUIRED_VARIABLES

In [ ]:
build_monthly <- function(ids) {

  ids <- unique(
    as.character(ids)
  )

  d <- load_raw_data()[
    artist_id %in% ids
  ]

  if (nrow(d) == 0) {

    return(
      data.table()
    )
  }

  d[
    ,
    variable := paste(
      endpoint,
      source,
      metric,
      sep = "__"
    )
  ]

  d <- d[
    variable %in%
      REQUIRED_VARIABLES
  ]

  d[
    ,
    timestamp_dt :=
      parse_timestamp(
        timestamp
      )
  ]

  d[
    ,
    observed_date :=
      as.Date(
        timestamp_dt
      )
  ]

  # Mismo período para TODOS los artistas,
  # incluso los descargados originalmente desde 2010.
  d <- d[
    !is.na(observed_date) &
      observed_date >= FAN_SINCE &
      observed_date <= END_DATE
  ]

  d[
    ,
    month := as.Date(
      format(
        observed_date,
        "%Y-%m-01"
      )
    )
  ]

  setorder(
    d,
    artist_id,
    endpoint,
    source,
    metric,
    month,
    timestamp_dt
  )

  monthly <- d[
    ,
    .SD[.N],
    by = .(
      artist_id,
      endpoint,
      source,
      metric,
      variable,
      month
    )
  ]

  monthly <- monthly[
    ,
    .(
      artist_id,
      month,
      endpoint,
      source,
      metric,
      variable,
      observed_date,
      value,
      interpolated,
      extrapolated,
      flags
    )
  ]

  if (nrow(ARTISTS) > 0) {

    monthly <- merge(
      monthly,
      ARTISTS,
      by.x = "artist_id",
      by.y = "ID",
      all.x = TRUE
    )

    setcolorder(
      monthly,
      c(
        "artist_id",
        "artist_name",
        "cm_artist_rank",
        setdiff(
          names(monthly),
          c(
            "artist_id",
            "artist_name",
            "cm_artist_rank"
          )
        )
      )
    )
  }

  monthly
}

In [ ]:
# ============================================================
# DATASET MENSUAL CONSOLIDADO
# ============================================================

load_monthly_store <- function() {

  x <- safe_read_rds(
    MONTHLY_FILE,
    default = data.table()
  )

  as.data.table(
    x
  )
}


initialize_monthly_store <- function(
  legacy_ids
) {

  if (
    file.exists(
      MONTHLY_FILE
    )
  ) {

    cat(
      "El mensual consolidado ya existe.\n"
    )

    return(
      invisible(NULL)
    )
  }

  cat(
    "Construyendo base mensual legacy...\n"
  )

  legacy_monthly <-
    build_monthly(
      legacy_ids
    )

  safe_save_rds(
    legacy_monthly,
    MONTHLY_FILE
  )

  cat(
    "Base mensual legacy creada:",
    nrow(
      legacy_monthly
    ),
    "filas\n"
  )

  invisible(NULL)
}

In [ ]:
# ============================================================
# BASE MENSUAL LEGACY (los 300 artistas originales)
# Sólo se construye si el mensual consolidado no existe.
# ============================================================

initialize_monthly_store(
  BASE_300_IDS
)

In [ ]:
# ============================================================
# DATASET MENSUAL CONSOLIDADO
# Fan Stats + Playlist Evolution
# ============================================================

consolidate_monthly_requests <- function(ids) {

  ids <- unique(
    as.character(ids)
  )


  # ----------------------------------------------------------
  # Archivos mensuales individuales de Fan Stats
  # ----------------------------------------------------------

  fan_files <- unlist(
    lapply(
      ids,
      function(id) {

        vapply(
          FAN_SOURCES,
          function(source) {

            monthly_response_path(
              fan_request_key(
                id,
                source
              )
            )
          },
          FUN.VALUE = character(1)
        )
      }
    ),
    use.names = FALSE
  )


  # ----------------------------------------------------------
  # Archivos mensuales individuales de Playlist Evolution
  # ----------------------------------------------------------

  playlist_files <- vapply(
    ids,
    function(id) {

      monthly_response_path(
        playlist_request_key(
          id
        )
      )
    },
    FUN.VALUE = character(1)
  )


  fan_files <- unique(
    fan_files[
      file.exists(fan_files)
    ]
  )

  playlist_files <- unique(
    playlist_files[
      file.exists(playlist_files)
    ]
  )


  cat(
    "Archivos mensuales Fan Stats encontrados:",
    length(fan_files),
    "\n"
  )

  cat(
    "Archivos mensuales Playlist encontrados:",
    length(playlist_files),
    "\n"
  )


  files <- unique(
    c(
      fan_files,
      playlist_files
    )
  )


  if (
    length(files) == 0
  ) {

    cat(
      "No hay archivos mensuales nuevos para consolidar.\n"
    )

    return(
      invisible(NULL)
    )
  }


  nuevos <- rbindlist(
    lapply(
      files,
      function(path) {

        safe_read_rds(
          path,
          default = data.table()
        )
      }
    ),
    fill = TRUE,
    use.names = TRUE
  )


  base <- load_monthly_store()


  all <- rbindlist(
    list(
      base,
      nuevos
    ),
    fill = TRUE,
    use.names = TRUE
  )


  # Asegurar que todas las filas tengan el nombre
  # estandarizado de la variable.

  if (
    !"variable" %in% names(all)
  ) {

    all[
      ,
      variable := paste(
        endpoint,
        source,
        metric,
        sep = "__"
      )
    ]

  } else {

    all[
      is.na(variable) |
        variable == "",
      variable := paste(
        endpoint,
        source,
        metric,
        sep = "__"
      )
    ]
  }


  if (
    "artist_name" %in%
      names(all)
  ) {

    all[
      ,
      artist_name := NULL
    ]
  }


  if (
    "cm_artist_rank" %in%
      names(all)
  ) {

    all[
      ,
      cm_artist_rank := NULL
    ]
  }


  all <- unique(
    all,
    by = c(
      "artist_id",
      "endpoint",
      "source",
      "metric",
      "month"
    ),
    fromLast = TRUE
  )


  all <- merge(
    all,
    ARTISTS,
    by.x = "artist_id",
    by.y = "ID",
    all.x = TRUE
  )


  setcolorder(
    all,
    c(
      "artist_id",
      "artist_name",
      "cm_artist_rank",
      setdiff(
        names(all),
        c(
          "artist_id",
          "artist_name",
          "cm_artist_rank"
        )
      )
    )
  )


  safe_save_rds(
    all,
    MONTHLY_FILE
  )


  cat(
    "Mensual consolidado:",
    nrow(all),
    "filas\n\n"
  )


  print(
    all[
      ,
      .(
        artistas = uniqueN(artist_id),
        variables = uniqueN(variable),
        filas = .N
      ),
      by = endpoint
    ][
      order(endpoint)
    ]
  )


  invisible(NULL)
}

### Fan Stats legacy sin mensual propio

Artistas con descarga completa en el log cuyos datos diarios quedaron sólo en el cache del piloto. La celda arma esos mensuales desde el cache, sin requests. Si no queda nada pendiente, no hace nada.

In [ ]:
# ============================================================
# FAN STATS LEGACY: MENSUAL DESDE EL CACHE DEL PILOTO
#
# Tareas artista-source de la muestra que:
# - figuran completas en el log;
# - no tienen archivo mensual propio;
# - no tienen Fan Stats de esa plataforma en el mensual
#   consolidado;
# - tienen datos en el cache del piloto (DATA_FILE).
# Se filtran las métricas objetivo (TARGET_FAN_METRICS), igual
# que parse_fan_stats, y el mensual se guarda con la clave
# original de la tarea (fan_request_key), así
# consolidate_monthly_requests() lo toma. No escribe en el log:
# la descarga ya figura completa.
# ============================================================

log_legacy <- read_request_log()

raw_legacy <- as.data.table(
  safe_read_rds(DATA_FILE, default = data.table())
)

raw_legacy <- raw_legacy[
  endpoint == "fan_stats" &
    artist_id %in% PILOT2_ARTIST_IDS
]

fan_stats_en_store <- unique(
  load_monthly_store()[endpoint == "fan_stats", .(artist_id, source)]
)[, en_store := TRUE]

tareas_legacy <- merge(
  unique(raw_legacy[, .(artist_id, source)]),
  fan_stats_en_store,
  by = c("artist_id", "source"),
  all.x = TRUE
)[
  is.na(en_store) &
    !file.exists(monthly_response_path(fan_request_key(artist_id, source)))
]

tareas_legacy <- tareas_legacy[
  mapply(
    function(id, src) fan_completed_tramos(log_legacy, id, src),
    artist_id,
    source
  )
]

cat(
  "Tareas legacy sin mensual propio:", nrow(tareas_legacy), "\n",
  "Artistas:", uniqueN(tareas_legacy$artist_id), "\n\n"
)

filas_mensuales_legacy <- integer()

for (i in seq_len(nrow(tareas_legacy))) {

  id <- tareas_legacy$artist_id[i]
  src <- tareas_legacy$source[i]

  raw <- unique(
    raw_legacy[
      artist_id == id &
        source == src &
        metric %in% TARGET_FAN_METRICS[[src]]
    ],
    by = c("metric", "timestamp")
  )

  monthly <- fan_raw_to_monthly(raw)

  save_monthly_response(fan_request_key(id, src), monthly)

  filas_mensuales_legacy[i] <- nrow(monthly)
}

if (nrow(tareas_legacy) > 0) {

  tareas_legacy[, filas_mensuales := filas_mensuales_legacy]

  print(
    tareas_legacy[
      ,
      .(tareas = .N, artistas = uniqueN(artist_id), filas_mensuales = sum(filas_mensuales)),
      by = source
    ]
  )
}

In [ ]:
# ============================================================
# CONSOLIDAR REQUESTS Y CONSTRUIR EL DATASET MENSUAL DEL PILOTO
# ============================================================

consolidate_monthly_requests(
  PILOT2_ARTIST_IDS
)

pilot2_monthly <- load_monthly_store()[
  artist_id %in% PILOT2_ARTIST_IDS
]

cat(
  "Artistas objetivo:",
  length(unique(PILOT2_ARTIST_IDS)),
  "\n"
)

cat(
  "Artistas con algún dato mensual:",
  uniqueN(pilot2_monthly$artist_id),
  "\n"
)

cat(
  "Filas mensuales:",
  nrow(pilot2_monthly),
  "\n"
)

cat(
  "Primer mes:",
  format(min(pilot2_monthly$month, na.rm = TRUE)),
  "\n"
)

cat(
  "Último mes:",
  format(max(pilot2_monthly$month, na.rm = TRUE)),
  "\n\n"
)

cat(
  "Contenido por endpoint:\n"
)

print(
  pilot2_monthly[
    ,
    .(
      artistas = uniqueN(artist_id),
      variables = uniqueN(variable),
      filas = .N
    ),
    by = endpoint
  ][order(endpoint)]
)

cat(
  "\nVariables objetivo todavía ausentes:\n"
)

print(
  setdiff(
    REQUIRED_VARIABLES,
    unique(pilot2_monthly$variable)
  )
)

# ------------------------------------------------------------
# Control: lo descargado sólo llega al mensual consolidado y al
# panel si el artista está en PILOT2_ARTIST_IDS
# ------------------------------------------------------------

ids_con_mensual <- unique(
  vapply(
    strsplit(
      list.files(MONTHLY_REQUEST_DIR, pattern = "\\.rds$"),
      "__",
      fixed = TRUE
    ),
    `[`,
    character(1),
    2
  )
)

con_mensual_fuera_de_muestra <- setdiff(ids_con_mensual, PILOT2_ARTIST_IDS)

con_mensual_fuera_en_marco <- intersect(
  con_mensual_fuera_de_muestra,
  construir_marco_top(rank_data, MARCO_TOP_N)$ID
)

muestra_sin_fan_stats <- setdiff(
  PILOT2_ARTIST_IDS,
  pilot2_monthly[endpoint == "fan_stats", unique(artist_id)]
)

cat(
  "\nControl de integración:\n",
  "Artistas con mensuales descargados fuera de la muestra:",
  length(con_mensual_fuera_de_muestra),
  "(dentro del marco:", length(con_mensual_fuera_en_marco), ")\n",
  "Artistas de la muestra sin Fan Stats en el mensual:",
  length(muestra_sin_fan_stats), "\n\n"
)

if (length(con_mensual_fuera_en_marco) > 0) {

  warning(
    "Hay artistas del marco con datos descargados que no integran ",
    "la muestra y no llegan al panel: ",
    paste(con_mensual_fuera_en_marco, collapse = ", ")
  )
}

head(pilot2_monthly)

## 13. Exportación

In [ ]:
# ============================================================
# GUARDAR EL DATASET MENSUAL (LARGO Y ANCHO)
#
# Antes, el CSV largo y el ancho se guardaban en dos celdas
# separadas con un número de artistas hardcodeado en el nombre
# de archivo ("850_..."), que había que actualizar a mano cada
# vez que crecía la muestra (de ahí los CSV de 600 y 740
# artistas que quedaron sueltos en el directorio). Esta función
# calcula el número de artistas real y arma ambos archivos
# (long y wide) a partir del mismo dataset, para no tener dos
# fuentes de verdad.
# Nombre: artistas_ST_raw_<n artistas>_long.csv / _wide.csv
# (con suffix: artistas_ST_raw_<n>_<suffix>_long.csv).
# ============================================================

export_monthly_dataset <- function(monthly, suffix = "") {

  n_artistas <- uniqueN(monthly$artist_id)

  tag <- if (nzchar(suffix)) {
    paste0("artistas_ST_raw_", n_artistas, "_", suffix)
  } else {
    paste0("artistas_ST_raw_", n_artistas)
  }

  long_file <- paste0(tag, "_long.csv")
  wide_file <- paste0(tag, "_wide.csv")

  fwrite(
    monthly,
    long_file
  )

  monthly_wide <- dcast(
    monthly,
    artist_id + artist_name + cm_artist_rank + month ~ variable,
    value.var = "value"
  )

  fwrite(
    monthly_wide,
    wide_file
  )

  cat(
    "Long:", long_file, "-",
    nrow(monthly), "filas x", ncol(monthly), "columnas\n"
  )

  cat(
    "Wide:", wide_file, "-",
    nrow(monthly_wide), "filas x", ncol(monthly_wide), "columnas\n"
  )

  invisible(
    list(
      long = monthly,
      wide = monthly_wide,
      long_file = long_file,
      wide_file = wide_file
    )
  )
}

pilot2_monthly_export <- export_monthly_dataset(pilot2_monthly)
pilot2_monthly_wide <- pilot2_monthly_export$wide

In [ ]:
# ============================================================
# INSUMOS DEL NOTEBOOK 02
# Muestra, catálogo, deciles y marco, para que el notebook de
# cobertura y panel no dependa de correr la extracción.
# Archivo local, excluido de git (licencia de Chartmetric).
# ============================================================

INSUMOS_NOTEBOOK_02_FILE <- file.path(
  OUT_DIR,
  "insumos_notebook_02.rds"
)

insumos_notebook_02 <- list(
  fecha_generacion = Sys.time(),
  fan_since = FAN_SINCE,
  end_date = END_DATE,
  required_variables = REQUIRED_VARIABLES,
  muestra_ids = as.character(PILOT2_ARTIST_IDS),
  artistas = ARTISTS,
  rank_deciles_7000 = rank_data[, .(ID = as.character(ID), cm_artist_rank, decile)],
  marco_top_n = MARCO_TOP_N,
  marco_top = construir_marco_top(rank_data, MARCO_TOP_N)
)

safe_save_rds(
  insumos_notebook_02,
  INSUMOS_NOTEBOOK_02_FILE
)

cat(
  "Insumos guardados en", INSUMOS_NOTEBOOK_02_FILE, "\n",
  "Artistas de la muestra:", length(insumos_notebook_02$muestra_ids), "\n",
  "Marco: top", MARCO_TOP_N, "-", nrow(insumos_notebook_02$marco_top), "artistas\n"
)